# V-Energy Grid 360 — Camada Gold (DE-03)

## 1. Visão Geral e Arquitetura da Camada Gold
A camada **Gold** representa o topo da arquitetura Medalhão. Sua responsabilidade é consolidar, agregar e enriquecer os dados limpos vindos da camada **Silver**, transformando tabelas relacionais em **visões analíticas de negócio**.

### Objetivos do Negócio Atendidos:
1. **Diagnóstico de Receita e Tarifa:** Responder à diretoria sobre a variação do volume faturado (-4%) versus o reajuste tarifário (+10,5%), identificando o impacto do consumo, bandeiras e Geração Distribuída (GD).
2. **Qualidade do Serviço (DEC/FEC):** Calcular indicadores internos simplificados de continuidade por alimentador e mês, sem alegar apuração ou conformidade regulatória oficial.
3. **Fila de Inspeção e Risco:** Identificar potenciais perdas não técnicas (fraudes/gatos) cruzando dados de consumo, inspeções anteriores e chamados do call center.
4. **Visão 360° da Unidade Consumidora:** Fonte única de verdade para alimentar o backend do Portal Web e o Agente de IA.

---

### Histórias de Usuário Atendidas neste Notebook:
* **Modelagem de Métricas:** *"Como engenheiro de dados, quero modelar a camada Gold a partir das perguntas de negócio das lideranças..."*
* **Indicadores ANEEL:** *"Como engenheiro de dados, quero calcular DEC e FEC por alimentador conforme a definição interna simplificada documentada..."*
* **Testes de Qualidade:** *"Como engenheiro de dados, quero testes de qualidade automatizados no pipeline..."*

###  Leitura de Controle e Rastreabilidade 

**Objetivo:** Capturar os metadados de execução da camada Silver antes do processamento.  
**Impacto:** Permite associar cada carga da Gold ao `run_id` e carimbo de data/hora (`data_processamento`) exato do lote de origem, garantindo auditoria completa entre camadas.

###  Governança de Ambiente e Parâmetros de Execução (`AMBIENTE`)

**Objetivo:** Garantir a integridade das travas de qualidade e a rastreabilidade entre as camadas Silver e Gold, sincronizando a execução do notebook entre ambientes de desenvolvimento (`dev`) e produção (`prod`).

---

####  Orientações de Execução para a Equipe:

1. **Execução Automatizada em Produção (Databricks Workflows / Job Oficial):**
   * O orquestrador injeta automaticamente o parâmetro `AMBIENTE = prod` no runtime do Job.
   * **Garantia de Produção:** O pipeline valida se a camada Silver possui o status `'CONCLUIDO'`, bloqueia execuções com dados sintéticos e registra a execução como oficial de auditoria na tabela `gold.controle_gold`.

2. **Execução Interativa Manual (Desenvolvimento e Validação no Notebook):**
   * Devido à persistência do estado dos widgets na interface visual do Databricks, verifique o widget **`Ambiente de Execução (AMBIENTE)`** no canto superior esquerdo do notebook.
   * **Para homologação manual com comportamento de Produção:** Certifique-se de que a caixinha do widget exibe `prod` antes de rodar todas as células.
   * **Caso a tela exiba `dev`:** Altere manualmente o valor da caixa para `prod`, aperte *Enter* e reexecute a **Seção 0**.

3. **Registro Oficial de Governança:**
   * Execuções manuais geram histórico de desenvolvimento. O carimbo oficial de homologação dos dados para consumo corporativo (Aplicações / BI) é o registrado pelas execuções do **Job Automatizado de Produção**.

In [0]:
# SEÇÃO 0 — CONFIGURAÇÃO, GOVERNANÇA E FUNÇÕES AUXILIARES

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


CATALOG = "workspace"
SCHEMA_SILVER_DADOS = "silver_publicada"
SCHEMA_SILVER_CONTROLE = "silver"
SCHEMA_GOLD = "gold"

# AMBIENTE deve vir dos parâmetros do Databricks Job/Workflow.
# Não atribuir "dev" ou "prod" manualmente em outra parte do notebook.
AMBIENTE = dbutils.widgets.get("AMBIENTE").strip().lower()

if AMBIENTE not in {"dev", "test", "prod"}:
    raise ValueError(
        f"AMBIENTE inválido: {AMBIENTE!r}. Use dev, test ou prod."
    )

PERMITIR_RUN_SINTETICO = AMBIENTE in {"dev", "test"}

# Alias para células anteriores que usam SCHEMA_SILVER.
SCHEMA_SILVER = SCHEMA_SILVER_DADOS

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")

print(f"[GOVERNANÇA] Ambiente recebido do Job: {AMBIENTE}")
print(f"[GOVERNANÇA] Permite Silver sintética: {PERMITIR_RUN_SINTETICO}")
print(f"Silver Dados:    {CATALOG}.{SCHEMA_SILVER_DADOS}")
print(f"Silver Controle: {CATALOG}.{SCHEMA_SILVER_CONTROLE}")
print(f"Gold:            {CATALOG}.{SCHEMA_GOLD}")


def assert_columns_exist(df, columns, name):
    """Falha com mensagem clara quando faltam colunas obrigatórias."""
    missing = sorted(set(columns) - set(df.columns))

    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")


def assert_no_nulls(df, columns, name):
    """Falha se houver nulos nas colunas informadas."""
    condition = None

    for column in columns:
        part = F.col(column).isNull()
        condition = part if condition is None else (condition | part)

    if condition is not None and df.filter(condition).limit(1).count() > 0:
        raise ValueError(
            f"{name}: há valores nulos nas colunas obrigatórias {columns}."
        )


def assert_unique(df, columns, name):
    """Falha se a chave informada não for única."""
    duplicates = (
        df.groupBy(*columns)
          .count()
          .filter(F.col("count") > 1)
          .limit(1)
          .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"{name}: a chave {columns} não é única. "
            "Revise a granularidade da fonte antes de continuar."
        )


def get_distinct_uc_with_gd(df_gd):
    """
    Retorna uma linha por UC com GD.

    A regra considera que qualquer registro em geracao_distribuida
    significa que a UC possui GD. Se a fonte tiver status ou vigência,
    substitua essa regra pela condição de negócio apropriada.
    """
    assert_columns_exist(df_gd, ["id_uc"], "geracao_distribuida")

    return (
        df_gd
        .filter(F.col("id_uc").isNotNull())
        .select("id_uc")
        .dropDuplicates(["id_uc"])
        .withColumn("possui_gd", F.lit("Sim"))
    )


# ----------------------------------------------------------------------
# A1 — Validação cruzada entre controle_publicacao e controle_silver
# ----------------------------------------------------------------------

TB_PUBLICACAO = (
    f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_publicacao"
)
TB_SILVER = (
    f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_silver"
)

# Inicializar as variáveis para que seus nomes existam em ambos os caminhos.
CONTROL_TABLE_PUBLICACAO = TB_PUBLICACAO
CONTROL_TABLE_SILVER = TB_SILVER
status_exec = None
silver_lineage_status = None

tabelas_controle_existentes = (
    spark.catalog.tableExists(TB_PUBLICACAO)
    and spark.catalog.tableExists(TB_SILVER)
)

if not tabelas_controle_existentes:
    if not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            "Falha crítica de governança: são necessárias as tabelas "
            f"{TB_PUBLICACAO} e {TB_SILVER}. "
            "A execução Gold foi interrompida."
        )

    agora_utc = datetime.now(timezone.utc)

    run_id_silver = (
        f"DEV_SYNTHETIC_{agora_utc.strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid4().hex[:8]}"
    )
    lote_bronze_silver = "DEV_LOTE"
    data_processamento_silver = agora_utc

    status_exec = "SEM_CONTROLE"
    silver_lineage_status = "SINTETICO_SEM_CONTROLE_SILVER"

    print("[AVISO DE GOVERNANÇA] Tabelas físicas de controle não encontradas.")
    print(f"   • Ambiente: {AMBIENTE}")
    print(f"   • Run ID sintético: {run_id_silver}")
    print(f"   • Lote sintético: {lote_bronze_silver}")
    print(f"   • Status Silver: {status_exec}")
    print(f"   • Linhagem: {silver_lineage_status}")
    print("   • O ID sintético não comprova a versão da Silver consumida.")

else:
    df_publicacao = spark.table(TB_PUBLICACAO)
    df_controle_silver = spark.table(TB_SILVER)

    # A publicação precisa informar o run e o lote ativos.
    assert_columns_exist(
        df_publicacao,
        ["run_ativo", "lote_ativo"],
        TB_PUBLICACAO
    )

    col_data_publicacao = next(
        (
            coluna
            for coluna in [
                "publicado_em",
                "data_processamento",
                "concluido_em"
            ]
            if coluna in df_publicacao.columns
        ),
        None
    )

    if col_data_publicacao is None:
        raise ValueError(
            f"{TB_PUBLICACAO}: não foi encontrada uma coluna de data "
            "entre publicado_em, data_processamento e concluido_em."
        )

    registros_publicacao = (
        df_publicacao
        .filter(
            F.col("run_ativo").isNotNull()
            & F.col("lote_ativo").isNotNull()
            & F.col(col_data_publicacao).isNotNull()
        )
        .withColumn(
            "_data_publicacao_ts",
            F.to_timestamp(F.col(col_data_publicacao))
        )
        .filter(F.col("_data_publicacao_ts").isNotNull())
        .orderBy(
            F.col("_data_publicacao_ts").desc(),
            F.col("run_ativo").cast("string").desc(),
            F.col("lote_ativo").cast("string").desc()
        )
        .limit(1)
        .collect()
    )

    if not registros_publicacao:
        raise ValueError(
            f"{TB_PUBLICACAO}: não há registro válido com run_ativo, "
            "lote_ativo e data de publicação."
        )

    registro_publicacao = registros_publicacao[0]

    run_id_silver = registro_publicacao["run_ativo"]
    lote_bronze_silver = registro_publicacao["lote_ativo"]
    data_processamento_silver = registro_publicacao["_data_publicacao_ts"]

    # Reconhecer os nomes de coluna usados pela tabela de histórico.
    colunas_silver = set(df_controle_silver.columns)

    col_run_silver = next(
        (
            coluna
            for coluna in ["_run_id", "run_id", "run_ativo"]
            if coluna in colunas_silver
        ),
        None
    )

    col_lote_silver = next(
        (
            coluna
            for coluna in ["_lote_bronze", "lote_bronze", "lote_ativo"]
            if coluna in colunas_silver
        ),
        None
    )

    col_status_silver = next(
        (
            coluna
            for coluna in ["status", "situacao"]
            if coluna in colunas_silver
        ),
        None
    )

    colunas_obrigatorias_ausentes = [
        descricao
        for descricao, coluna in [
            ("coluna de run_id", col_run_silver),
            ("coluna de lote Bronze", col_lote_silver),
            ("coluna de status", col_status_silver)
        ]
        if coluna is None
    ]

    if colunas_obrigatorias_ausentes:
        raise ValueError(
            f"{TB_SILVER}: schema inválido; faltam "
            f"{', '.join(colunas_obrigatorias_ausentes)}."
        )

    # Procurar todos os registros que correspondem ao run e lote publicados.
    df_status_run_lote = (
        df_controle_silver
        .filter(
            (F.col(col_run_silver).cast("string") == str(run_id_silver))
            & (F.col(col_lote_silver).cast("string") == str(lote_bronze_silver))
        )
        .withColumn(
            "_status_norm",
            F.upper(F.trim(F.col(col_status_silver).cast("string")))
        )
    )

    resumo_status = (
        df_status_run_lote
        .agg(
            F.count(F.lit(1)).alias("qtd_registros"),
            F.sum(
                F.when(F.col("_status_norm").isNull(), 1).otherwise(0)
            ).alias("qtd_status_nulos"),
            F.collect_set("_status_norm").alias("status_encontrados")
        )
        .first()
    )

    qtd_registros = resumo_status["qtd_registros"] or 0
    qtd_status_nulos = resumo_status["qtd_status_nulos"] or 0
    status_encontrados = resumo_status["status_encontrados"] or []

    status_concluido = (
        qtd_registros > 0
        and qtd_status_nulos == 0
        and all(
            status == "CONCLUIDO"
            for status in status_encontrados
        )
    )

    # Esta variável é consumida pela célula final controle_gold.
    if status_concluido:
        status_exec = "CONCLUIDO"
    elif qtd_registros == 0:
        status_exec = "SEM_REGISTRO"
    elif qtd_status_nulos > 0:
        status_exec = "STATUS_NULO"
    else:
        status_exec = ", ".join(sorted(status_encontrados))

    if not status_concluido and not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            "Falha de execução Silver: "
            f"run={run_id_silver}, "
            f"lote={lote_bronze_silver}, "
            f"status={status_exec!r}, "
            f"registros={qtd_registros}, "
            f"status nulos={qtd_status_nulos}. "
            "A Gold exige status CONCLUIDO."
        )

    if status_concluido:
        silver_lineage_status = "CONTROLE_FISICO"
        print("[CONTROLE SILVER] Validação cruzada concluída.")
    else:
        silver_lineage_status = "CONTROLE_FISICO_NAO_CONCLUIDO"
        print(
            "[AVISO DE GOVERNANÇA] Controle físico encontrado, "
            "mas o status não foi aprovado."
        )

    print(f"   • Publicação: {TB_PUBLICACAO}")
    print(f"   • Histórico: {TB_SILVER}")
    print(f"   • Run ID: {run_id_silver}")
    print(f"   • Lote: {lote_bronze_silver}")
    print(f"   • Processada em: {data_processamento_silver}")
    print(f"   • Status Silver: {status_exec}")
    print(f"   • Linhagem: {silver_lineage_status}")


# Função auxiliar para flag de Geração Distribuída

def get_distinct_uc_with_gd(df_gd):
    """
    Retorna uma linha por UC com GD.

    Regra atual: qualquer registro em geracao_distribuida indica GD.
    Se houver status ou vigência na fonte, aplique aqui a regra aprovada.
    """
    assert_columns_exist(
        df_gd,
        ["id_uc"],
        "geracao_distribuida"
    )

    return (
        df_gd
        .filter(F.col("id_uc").isNotNull())
        .select("id_uc")
        .dropDuplicates(["id_uc"])
        .withColumn("possui_gd", F.lit("Sim"))
    )

## 2. Diagnóstico de Receita e Variação Tarifária
### Objetivo:
Atender ao pedido do Diretor Comercial explicando a queda no volume faturado e o impacto na receita por município, classe de consumo, subgrupo e presença de geração solar (GD).

### Regras de Negócio e Agregações:
* Cruzamento das tabelas `silver.faturas`, `silver.leituras`, `silver.unidades_consumidoras` e `silver.geracao_distribuida`.
* Métricas geradas: Consumo total , Receita total , Ticket médio e Tarifa Média Efetiva.

In [0]:

# SEÇÃO 1: Construção da Gold Diagnóstico de Receita

df_faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
df_leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
df_ucs = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")
df_gd = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida")

# Validar colunas esperadas nas fontes
assert_columns_exist(df_faturas, ["id_uc", "mes_referencia", "valor_total"], "faturas")
assert_columns_exist(df_leituras, ["id_uc", "mes_referencia", "consumo_kwh"], "leituras")
assert_columns_exist(df_ucs, ["id_uc", "municipio", "classe", "subgrupo_tarifario"], "unidades_consumidoras")

# Base de UCs (uma linha de cadastro por UC, removendo UCs sintéticas de teste)
df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .select("id_uc", "municipio", "classe", "subgrupo_tarifario")
)

assert_unique(df_ucs_base, ["id_uc"], "unidades_consumidoras")

# Agregar faturas por UC e mês
df_faturas_uc_mes = (
    df_faturas
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("valor_total").alias("receita_uc_mes"),
        F.count(F.lit(1)).alias("qtd_faturas_uc_mes")
    )
)

assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "faturas agregadas por UC e competência")

# Agregar leituras por UC e mês
df_leituras_uc_mes = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("consumo_kwh").alias("consumo_uc_mes"),
        F.count(F.col("consumo_kwh")).alias("qtd_leituras_validas_uc_mes")
    )
)

assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "leituras agregadas por UC e competência")

# Unir faturas com leituras e cadastro (INNER JOIN entre faturas e leituras)
df_uc_mes = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "inner")
    .join(df_ucs_base, "id_uc", "left")
)

# Tratar bandeira tarifária se existir em faturas
if "bandeira_tarifaria" in df_faturas.columns:
    df_bandeira_uc_mes = (
        df_faturas
        .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
        .groupBy("id_uc", "mes_referencia")
        .agg(
            F.first("bandeira_tarifaria", ignorenulls=True).alias("bandeira_tarifaria"),
            F.countDistinct("bandeira_tarifaria").alias("qtd_bandeiras_uc_mes")
        )
    )

    df_uc_mes = df_uc_mes.join(df_bandeira_uc_mes, ["id_uc", "mes_referencia"], "left")

    if df_bandeira_uc_mes.filter(F.col("qtd_bandeiras_uc_mes") > 1).limit(1).count() > 0:
        raise ValueError(
            "Há mais de uma bandeira tarifária por UC/competência. "
            "Defina a regra de rateio ou a granularidade correta antes de gerar diagnostico_receita."
        )
else:
    df_uc_mes = df_uc_mes.withColumn("bandeira_tarifaria", F.lit(None).cast("string"))

# Adicionar Flag de GD
df_gd_uc = get_distinct_uc_with_gd(df_gd)

df_uc_mes = (
    df_uc_mes
    .join(df_gd_uc, "id_uc", "left")
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd"), F.lit("Não")))
)

# Métricas agregadas no grão de negócio (Competência/Município/Classe/Subgrupo/Bandeira/GD)
group_columns = [
    "mes_referencia",
    "municipio",
    "classe",
    "subgrupo_tarifario",
    "bandeira_tarifaria",
    "possui_gd"
]

gold_diagnostico_receita = (
    df_uc_mes
    .groupBy(*group_columns)
    .agg(
        F.countDistinct("id_uc").alias("qtd_ucs_distintas_faturadas_no_mes"), # Nomenclatura explícita
        F.sum("qtd_faturas_uc_mes").cast("long").alias("soma_faturamentos_mensais"),                  # Registra volume de faturas do grupo
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.avg("receita_uc_mes").alias("ticket_medio_reais_por_uc"),
        F.avg("consumo_uc_mes").alias("consumo_medio_kwh_por_uc")
    )
    .withColumn(
        "tarifa_media_efetiva",
        F.when(
            F.col("consumo_total_kwh").isNotNull() & (F.col("consumo_total_kwh") != 0),
            F.col("receita_total_reais") / F.col("consumo_total_kwh")
        )
    )
    .withColumnRenamed("mes_referencia", "competencia")
    .withColumnRenamed("classe", "classe_consumo")
)

# Checagens antes da gravação
assert_no_nulls(gold_diagnostico_receita, ["competencia"], "diagnostico_receita")

if gold_diagnostico_receita.filter((F.col("receita_total_reais") < 0) | (F.col("consumo_total_kwh") < 0)).limit(1).count() > 0:
    raise ValueError(
        "diagnostico_receita: foram encontrados valores negativos em receita ou consumo. Revise a regra para estornos e ajustes."
    )

(
    gold_diagnostico_receita
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
)

print("Tabela 'diagnostico_receita' gerada e gravada com sucesso!")

###  Clarificação e Renomeação de Métricas Volumétricas (`diagnostico_receita`)

Em atendimento às revisões finais da camada Gold:

1. **Renomeação de Colunas:**
   * Ajustada a nomenclatura da coluna de acúmulo volumétrico de `qtd_ucs_faturadas` / `ucs_somadas_na_gold` para **`qtd_ucs_distintas_faturadas_no_mes`** e adicionada a **`soma_faturamentos_mensais`**.
2. **Eliminação de Ambiguidade:**
   * Como a tabela `diagnostico_receita` é agregada no grão de Competência/Município/Classe, a contagem de UCs é calculada na granularidade mensal. A nova nomenclatura elimina qualquer ambiguidade relacional entre o acúmulo mensal cumulativo e o total absoluto de UCs únicas cadastradas na rede.
3. **Exclusão de Registros Sintéticos:**
   * Aplicado o filtro de segurança na origem (`id_uc NOT LIKE 'TESTE%'`) para evitar a contaminação de faturamentos fictícios nos indicadores agregados de receita.

## 3. Indicadores Internos Simplificados de Continuidade (DEC / FEC)
### Objetivo:
Calcular a **Duração Equivalente de Interrupção por Unidade Consumidora (DEC)** e a **Frequência Equivalente de Interrupção por Unidade Consumidora (FEC)** agregadas por alimentador e competência, com município como atributo cadastral.

### Definição interna simplificada (sem comprovação de conformidade regulatória):
* **DEC (Duração Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{DEC} = \frac{\text{Soma das Horas de Interrupção por UC Afetada}}{\text{Total de UCs do Alimentador}}$$

* **FEC (Frequência Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{FEC} = \frac{\text{Quantidade de Pares UC-Ocorrência}}{\text{Total de UCs do Alimentador}}$$

O denominador usa o cadastro atual. São publicados somente grupos com ocorrência incluída; não é criada uma grade de zeros para grupos sem ocorrência.


In [0]:
# SEÇÃO 2: Construção da Gold DEC e FEC

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "data_abertura", "duracao_min"],
    "ocorrencias_rede"
)

assert_columns_exist(
    df_ucs,
    ["id_uc", "alimentador", "municipio"],
    "unidades_consumidoras"
)

# Uma linha cadastral por UC é necessária para atribuir alimentador e município.
# Se unidades_consumidoras for histórica, selecione o registro vigente
# para cada UC antes desta validação.
df_ucs_rede = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .filter(~F.col("id_uc").startswith("TESTE"))
    .select(
        "id_uc",
        F.col("alimentador").alias("id_alimentador"),
        "municipio"
    )
)

assert_unique(df_ucs_rede, ["id_uc"], "cadastro de UCs para DEC/FEC")

assert_no_nulls(
    df_ucs_rede,
    ["id_alimentador"],
    "cadastro de UCs para DEC/FEC"
)

# Município é mantido como atributo do alimentador, não como dimensão
# que alteraria a soma e o denominador do indicador.
df_municipio_alimentador = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("municipio").alias("qtd_municipios"),
        F.first("municipio", ignorenulls=True).alias("municipio")
    )
)

if (
    df_municipio_alimentador
    .filter(F.col("qtd_municipios") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há alimentadores associados a mais de um município. "
        "Defina uma regra para o município exibido ou remova essa "
        "dimensão da tabela de indicadores."
    )

# Validar campos essenciais e duração.
df_ocorrencias_validas = (
    df_ocorrencias
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & F.col("data_abertura").isNotNull()
    )
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "ocorrencias_rede contém durações negativas."
    )

# Cada evento por UC deve aparecer uma única vez.
assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

df_ocorrencias_prep = (
    df_ocorrencias_validas
    .withColumn(
        "competencia",
        F.date_format(F.to_timestamp("data_abertura"), "yyyy-MM")
    )
    .withColumn(
        "duracao_horas",
        F.col("duracao_min").cast("double") / F.lit(60.0)
    )
)

# Associar alimentador à ocorrência. O cadastro já foi validado como uma linha por UC.
df_ocorrencias_rede = (
    df_ocorrencias_prep
    .join(df_ucs_rede, "id_uc", "inner")
)

# Numerador por alimentador e competência: duração acumulada e número de eventos UC-ocorrência.
df_numeradores = (
    df_ocorrencias_rede
    .groupBy("id_alimentador", "competencia")
    .agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(
            F.struct("id_uc", "id_ocorrencia")
        ).alias("frequencia_total_interrupcoes"),
        F.sum("duracao_horas")
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Denominador cadastral atual por alimentador.
df_denominadores = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("id_uc").alias("total_ucs_alimentador")
    )
)

gold_indicadores_dec_fec = (
    df_numeradores
    .join(df_denominadores, "id_alimentador", "left")
    .join(df_municipio_alimentador.select("id_alimentador", "municipio"),
          "id_alimentador", "left")
    .withColumn(
        "DEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("duracao_total_interrupcoes_horas")
            / F.col("total_ucs_alimentador")
        )
    )
    .withColumn(
        "FEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("frequencia_total_interrupcoes")
            / F.col("total_ucs_alimentador")
        )
    )
    .select(
        "id_alimentador",
        "municipio",
        "competencia",
        "total_ucs_alimentador",
        "ucs_interrompidas",
        "frequencia_total_interrupcoes",
        "duracao_total_interrupcoes_horas",
        "DEC",
        "FEC"
    )
)

assert_no_nulls(
    gold_indicadores_dec_fec,
    ["id_alimentador", "competencia", "total_ucs_alimentador"],
    "indicadores_dec_fec"
)

if (
    gold_indicadores_dec_fec
    .filter(
        (F.col("total_ucs_alimentador") <= 0)
        | (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec: denominador inválido ou métrica negativa."
    )

(
    gold_indicadores_dec_fec
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
    )
)

print("Tabela indicadores_dec_fec gravada.")

###  Ajuste N2 — Exclusão de UCs de Teste no Denominador de Indicadores (DEC / FEC)

**Descrição da Mudança:**  
Aplicado o filtro `~F.col("id_uc").startswith("TESTE")` no DataFrame `df_ucs_rede`. 

**Motivação & Impacto:**  
Anteriormente, o cadastro continha **200 UCs de teste** (prefixo `TESTE`) que inflavam o denominador total do sistema de **45.000 para 45.200 UCs**. 
Ao incluir essas UCs fictícias no cálculo dos indicadores da rede, o denominador ficava subestimado, gerando distorções de até `0,185 h/UC` no **DEC** e `0,038` no **FEC** em determinados alimentadores.

**Validação do Contrato:**
* **Total de UCs ativas na rede:** `45.000` UCs cravadas (removidas as 200 de teste).
* **UCs de teste restantes:** `0`.

## 4. Fila de Inspeção e Risco de Perdas Não Técnicas

### Objetivo:
Consolidar indicadores de risco e anomalias de consumo por Unidade Consumidora (UC) para priorização das equipes de fiscalização de campo no combate a fraudes e furtos de energia.

---

###  Governança de Dados, Privacidade (LGPD) e Saneamento Cadastral:
Em conformidade estrita com as diretrizes de governança (D20), a LGPD e a arquitetura analítica consumida pelo portal do atendente (**WEB-03**) e pelos serviços de IA:

1. **Exclusão de Registros Sintéticos e de Teste:**
   * **Filtro na Origem:** Remoção de UCs de desenvolvimento e testes diretamente na base cadastral (`df_ucs_base`), filtrando identificadores fictícios (`id_uc.startswith("TESTE")`) ; o filtro Gold não verifica documentos.
   * **Efeito Cascata:** Como a `fila_inspecao` e a `visao_360_uc` derivam da `df_ucs_base`, o filtro cadastral exclui IDs TESTE dessas visões, sem provar sozinho que todos os demais registros representem UCs operacionais reais.

2. **Minimização de Dados:**
   * **Eliminação de Dados Pessoais:** Exclusão total de `cpf_cnpj` e `nome_titular` no `select` final, descartando o uso de mascaramento em prol do corte definitivo de PII para consumo analítico e de IA.
   * **Isolamento da Autenticação:** A validação de login e acesso continua sob responsabilidade exclusiva do banco relacional operado no backend (OLTP), enquanto a camada Gold (OLAP) mantém a chave (`id_uc`) associável ao cadastro autorizado, sem garantia de anonimização total.
   * **Descarte de Endereços e Metadados Técnicos:** Supressão de `logradouro` (endereço), CEP, coordenadas geográficas (`latitude`/`longitude`) e colunas de controle do ETL iniciadas em `_` (ex: `_lote_id`, `_run_id`).

3. **Validação de Sanidade e Integridade:**
   * Atualização das asserções de qualidade (`assert_no_nulls`) para assegurar a consistência e ausência de nulos apenas nos atributos mantidos na camada Gold (`id_uc`, `prioridade_inspecao`, `municipio`, `bairro` e `alimentador`).

---

###  Regra de Priorização Dinâmica (`prioridade_inspecao`):
A classificação da esteira de vistoria segue a hierarquia de risco abaixo, cobrindo 100% dos cenários operacionais da base:

1. **`ALTA - HISTORICO IRREGULAR`**: UCs com histórico prévio de irregularidades confirmadas em vistorias anteriores (`total_inspecoes_irregulares > 0`).
2. **`REVISAR - SEM HISTORICO DE CONSUMO`**: UCs que não possuem registros de medição de consumo na base histórica (`media_consumo_historico IS NULL`).
3. **`ALTA - CONSUMO ZERO`**: UCs sem geração distribuída/solar conectada (`possui_geracao_solar = 'Não'`) que apresentaram média de consumo zerada (`media_consumo_historico = 0`).
4. **`MEDIA - NUNCA INSPECIONADO`**: UCs sem inspeção registrada na base de fiscalização de campo (`total_inspecoes_realizadas = 0`).
5. **`BAIXA`**: Demais UCs que não atendem às condições anteriores; a categoria não comprova regularidade e não usa a variação de consumo.

---

###  Schema Homologado (14 Colunas):
1. **Identificador:** `id_uc`
2. **Localização Territorial:** `municipio`, `bairro`, `uf`
3. **Classificação Comercial:** `classe`, `subgrupo` *(renomeado de subgrupo_tarifario)*, `alimentador`
4. **Atributos Operacionais:** `data_ligacao`, `status`, `possui_geracao_solar` *(renomeado de possui_gd)*
5. **Métricas de Inspeção e Consumo:** `total_inspecoes_realizadas`, `total_inspecoes_irregulares`, `media_consumo_historico`
6. **Métrica de Decisão:** `prioridade_inspecao`

In [0]:
# SEÇÃO 3: Construção da Gold Fila de Inspeção

df_inspecoes = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.inspecoes"
)

df_atendimentos = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.atendimentos"
)

df_leituras = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.leituras"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

df_gd = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida"
)

assert_columns_exist(
    df_inspecoes,
    ["id_uc", "id_inspecao", "resultado"],
    "inspecoes"
)

assert_columns_exist(
    df_atendimentos,
    ["id_uc", "protocolo", "nota_satisfacao"],
    "atendimentos"
)

assert_columns_exist(
    df_leituras,
    ["id_uc", "consumo_kwh"],
    "leituras"
)

assert_columns_exist(
    df_ucs,
    ["id_uc"],
    "unidades_consumidoras"
)

# Filtro da base cadastral de UCs (removendo UCs de teste)
df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .dropDuplicates(["id_uc"])
)

assert_unique(df_ucs_base, ["id_uc"], "base da fila de inspeção")

# Validação e agregação das inspeções
df_inspecoes_validas = df_inspecoes.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_inspecao").isNotNull()
)

assert_unique(
    df_inspecoes_validas,
    ["id_inspecao"],
    "inspecoes por id_inspecao"
)

df_insp_summary = (
    df_inspecoes_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_inspecao")
         .alias("total_inspecoes_realizadas"),
        F.countDistinct(
            F.when(
                F.col("resultado") == "IRREGULARIDADE CONFIRMADA",
                F.col("id_inspecao")
            )
        ).alias("total_inspecoes_irregulares")
    )
)

# Agregação dos atendimentos de call center
df_atend_validos = df_atendimentos.filter(
    F.col("id_uc").isNotNull()
    & F.col("protocolo").isNotNull()
)

df_atend_summary = (
    df_atend_validos
    .groupBy("id_uc")
    .agg(
        F.countDistinct("protocolo")
         .alias("total_chamados_callcenter"),
        F.avg("nota_satisfacao")
         .alias("media_satisfacao_atendimento")
    )
)

# Média de consumo histórico por UC
df_consumo_medio = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("consumo_kwh").isNotNull()
    )
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_kwh")
         .alias("media_consumo_historico")
    )
)

df_gd_uc = get_distinct_uc_with_gd(df_gd)

# Construção bruta da fila com regras de prioridade
gold_fila_raw = (
    df_ucs_base
    .join(df_consumo_medio, "id_uc", "left")
    .join(df_insp_summary, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_gd_uc, "id_uc", "left")
    .withColumn(
        "total_inspecoes_realizadas",
        F.coalesce(F.col("total_inspecoes_realizadas"), F.lit(0))
    )
    .withColumn(
        "total_inspecoes_irregulares",
        F.coalesce(F.col("total_inspecoes_irregulares"), F.lit(0))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "possui_geracao_solar",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "prioridade_inspecao",
        F.when(
            F.col("total_inspecoes_irregulares") > 0,
            "ALTA - HISTORICO IRREGULAR"
        )
        .when(
            F.col("media_consumo_historico").isNull(),
            "REVISAR - SEM HISTORICO DE CONSUMO"
        )
        .when(
            (F.col("media_consumo_historico") == 0)
            & (F.col("possui_geracao_solar") == "Não"),
            "ALTA - CONSUMO ZERO"
        )
        .when(
            F.col("total_inspecoes_realizadas") == 0,
            "MEDIA - NUNCA INSPECIONADO"
        )
        .otherwise("BAIXA")
    )
)

# MODIFICAÇÃO: Projeção limpa de colunas sem PII, sem endereço e sem metadados internos (_)
gold_fila_inspecao = (
    gold_fila_raw
    .select(
        "id_uc",
        "municipio",
        "bairro",
        "uf",
        "classe",
        F.col("subgrupo_tarifario").alias("subgrupo"),
        "alimentador",
        "data_ligacao",
        "status",
        "possui_geracao_solar",
        "total_inspecoes_realizadas",
        "total_inspecoes_irregulares",
        "media_consumo_historico",
        "prioridade_inspecao"
    )
)

# Validação de unicidade por UC
assert_unique(
    gold_fila_inspecao,
    ["id_uc"],
    "gold fila_inspecao"
)

# MODIFICAÇÃO: Validação de não-nulos apenas nas chaves técnicas e na prioridade
assert_no_nulls(
    gold_fila_inspecao,
    ["id_uc", "prioridade_inspecao", "municipio", "bairro", "alimentador"],
    "gold fila_inspecao"
)

# Gravação na camada Gold
(
    gold_fila_inspecao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"
    )
)

print("Tabela fila_inspecao gravada com sucesso sem PII, endereços ou metadados de controle.")

##  SEÇÃO 5: Construção e Governança da Gold Visão 360 (`visao_360_uc`)

Esta seção consolida as métricas operacionais, comerciais, de atendimento e interrupções em nível de **Unidade Consumidora (UC)**.

####  Governança de Dados & Conformidade (WEB-03 / IA):
Em alinhamento com o time de frontend e os requisitos da equipe de dados, a tabela final foi projetada com escopo estrito de colunas para garantir **privacidade e rastreabilidade**:
* **Descarte de Dados Pessoais (PII):** Remoção completa de `cpf_cnpj` e `nome_titular`.
* **Descarte de Dados de Localização Extensa:** Remoção de `logradouro` (endereço), CEP e coordenadas geográficas (`latitude`/`longitude`).
* **Exclusão de Metadados Técnicos:** Supressão de colunas de controle do ETL iniciadas em `_` (ex: `_lote_id`, `_run_id`, `_geo_invalida`).

####  Schema Homologado (15 Colunas):
1. **Chave Primária:** `id_uc`
2. **Localização Territorial:** `municipio`, `bairro`, `uf`
3. **Classificação Comercial:** `classe`, `subgrupo` *(renomeado de subgrupo_tarifario)*, `alimentador`
4. **Atributos Cadastrais:** `data_ligacao`, `status`, `possui_gd`
5. **Métricas de Atendimento:** `total_chamados_callcenter`, `media_satisfacao_atendimento`
6. **Métricas de Interrupção:** `total_interrupcoes_sofridas`, `duracao_total_interrupcoes_horas`
7. **Métricas de Consumo:** `media_consumo_historico`

In [0]:
# SEÇÃO 4: Construção da Gold Visão 360

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "duracao_min"],
    "ocorrencias_rede"
)

# Uma linha por ocorrência sofrida por UC.
df_ocorrencias_validas = df_ocorrencias.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_ocorrencia").isNotNull()
)

assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError("ocorrencias_rede contém durações negativas.")

df_interrupcao_agg = (
    df_ocorrencias_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_ocorrencia")
         .alias("total_interrupcoes_sofridas"),
        F.sum(F.col("duracao_min").cast("double") / F.lit(60.0))
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Construção da Visão 360 agregando as fontes do modelo
gold_visao_360_raw = (
    df_ucs_base
    .join(df_gd_uc, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_interrupcao_agg, "id_uc", "left")
    .join(df_consumo_medio, "id_uc", "left")
    .withColumn(
        "possui_gd",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "total_interrupcoes_sofridas",
        F.coalesce(F.col("total_interrupcoes_sofridas"), F.lit(0))
    )
    .withColumn(
        "duracao_total_interrupcoes_horas",
        F.coalesce(
            F.col("duracao_total_interrupcoes_horas"),
            F.lit(0.0)
        )
    )
)

# Projeção limpa e explícita: descarta PII, endereços e metadados de engenharia
gold_visao_360 = (
    gold_visao_360_raw
    .select(
        "id_uc",
        "municipio",
        "bairro",
        "uf",
        "classe",
        F.col("subgrupo_tarifario").alias("subgrupo"),
        "alimentador",
        "data_ligacao",
        "status",
        "possui_gd",
        "total_chamados_callcenter",
        "media_satisfacao_atendimento",
        "total_interrupcoes_sofridas",
        "duracao_total_interrupcoes_horas",
        "media_consumo_historico"
    )
)

# Validação de unicidade por UC
assert_unique(gold_visao_360, ["id_uc"], "gold visao_360_uc")

# Validação de não-nulos em identificadores e chaves territoriais
assert_no_nulls(
    gold_visao_360, 
    ["id_uc", "municipio", "bairro", "alimentador"], 
    "gold visao_360_uc"
)

# Gravação na camada Gold
(
    gold_visao_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
)

print("Tabela visao_360_uc gravada com sucesso sem PII, endereços ou metadados de controle.")

## 6. Novas Tabelas/Visões Temporais para GenAI/Dev (Issue #40)

Nesta seção, implementamos as novas tabelas agregadas projetadas para atender às consultas do agente de IA e backend, permitindo análises granulares de consumo e receita por localização e variação temporal por Unidade Consumidora (UC).

---

###  Tabela 1: Consumo e Receita por Bairro e Mês (`consumo_receita_bairro`)

**Objetivo:** Permitir que a IA responda a perguntas executivas sobre desempenho territorial (ex.: *"Qual bairro mais perdeu volume de consumo?"* ou *"Qual o bairro com maior receita no mês?"*) sem a necessidade de varrer toda a série temporal individual das UCs.

* **Grão dos Dados:** Competência (`competencia`) × Município (`municipio`) × Bairro (`bairro`) × Classe de Consumo (`classe_consumo`).
* **Origem das Informações:** Cruzamento da base de faturamento/leituras válidas da Silver com a base cadastral de UCs.
* **Fórmula / Lógica das Métricas:**
  - `consumo_total_kwh`: $\sum(\text{consumo\_kwh})$ faturado na competência/bairro/classe.
  - `receita_total_reais`: $\sum(\text{valor\_total})$ faturado na competência/bairro/classe.
  - `qtd_ucs_distintas_faturadas_no_mes`: $countDistinct(\text{id\_uc})$ ativas/faturadas no agrupamento do mês.
* **Critério de Aceite e Conciliação:** A soma consolidada de `receita_total_reais` agrupada por competência e classe deve corresponder exatamente ao total da tabela `diagnostico_receita`.

#### Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `competencia` | String / Date | Mês e ano de referência da fatura (formato YYYY-MM) |
| `municipio` | String | Nome do município onde a UC está localizada |
| `bairro` | String | Nome do bairro associado à UC |
| `classe_consumo` | String | Classe de consumo da UC (Residencial, Comercial, Industrial, etc.) |
| `consumo_total_kwh` | Double | Soma do consumo em kWh faturado no mês/bairro/classe |
| `receita_total_reais` | Double | Soma da receita em Reais (R$) faturada no mês/bairro/classe |
| `qtd_ucs_distintas_faturadas_no_mes` | Long | Contagem de UCs únicas faturadas naquela combinação |


In [0]:
# Seção 5.1: tabela 1 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: consumo_receita_bairro ===")

# Confirmar a granularidade das fontes pré-agregadas.
assert_unique(
    df_faturas_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_faturas_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# Garantir uma linha cadastral por UC. Não deduplicar silenciosamente:
# se houver duplicatas, revise a regra de vigência do cadastro.
df_ucs_bairro = (
    df_ucs_base
    .select(
        "id_uc",
        "municipio",
        "bairro",
        F.col("classe").alias("classe_consumo")
    )
)

assert_unique(df_ucs_bairro, ["id_uc"], "cadastro de UCs para consumo por bairro")

# Agregar faturas e leituras já agregadas por UC/mês.
# O cadastro usa LEFT JOIN para não eliminar faturas elegíveis sem dimensão cadastral.
df_consumo_receita_bairro_base = (
    df_faturas_uc_mes
    .join(
        df_leituras_uc_mes,
        ["id_uc", "mes_referencia"],
        "inner"
    )
    .join(df_ucs_bairro, "id_uc", "left")
    .withColumn(
        "municipio",
        F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "bairro",
        F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "classe_consumo",
        F.coalesce(F.col("classe_consumo").cast("string"), F.lit("Não informado"))
    )
)

gold_consumo_receita_bairro = (
    df_consumo_receita_bairro_base
    .groupBy(
        F.col("mes_referencia").alias("competencia"),
        "municipio",
        "bairro",
        "classe_consumo"
    )
    .agg(
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.countDistinct("id_uc")
         .alias("qtd_ucs_distintas_faturadas_no_mes")
    )
)

assert_no_nulls(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

assert_unique(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

(
    gold_consumo_receita_bairro
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
)

print("Tabela consumo_receita_bairro gravada na Gold.")

display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
         .orderBy("competencia", "municipio", "bairro", "classe_consumo")
         .limit(10)
)

#### Registros de Auditoria e Reconciliação — Tabela 1

 * **Metodologia de Validação:** Foi realizado o somatório do campo `receita_total_reais` agrupado na tabela `consumo_receita_bairro` e confrontado contra a tabela de controle `gold_diagnostico_receita`.
 * **Resultado do Teste de Reconciliação:**
   - **Diferença / Divergência:** R$ 0,00 (0,00%)
 * **Status:** **APROVADO**. A conciliação financeira cravou em 100%, garantindo integridade total de dados e ausência de duplicações na junção entre faturas, leituras e cadastro de UCs.

###  Tabela 2: Variação de Consumo por UC (`variacao_consumo_uc`)

**Objetivo:** Permitir que a IA e os modelos analíticos respondam a perguntas sobre comportamento temporal e anomalias de consumo individualizado (ex.: *"Quais UCs com painel solar tiveram queda de consumo acima de 30%?"*).

---

####  Regra de Negócio & Tratamento de Histórico:
* **Grão dos Dados:** Unidade Consumidora (`id_uc`).
* **Período Inicial (P1):** Média de consumo mensal de 2022-07 a 2023-06.
* **Período Final (P2):** Média de consumo mensal de 2025-07 a 2026-06.
* **Fórmula da Variação Absoluta (`variacao_kwh`):** $\text{Consumo Médio Final} - \text{Consumo Médio Inicial}$
* **Fórmula da Variação Percentual (`variacao_pct`):** $\frac{\text{Consumo Médio Final} - \text{Consumo Médio Inicial}}{\text{Consumo Médio Inicial}} \times 100$
* **Tratamento de Nulos & Flexibilidade de Histórico:** A variação percentual (`variacao_pct`) fica estritamente **nula (`NULL`)** apenas quando a UC não possui histórico de leitura em **nenhum mês** dos períodos analisados (P1 e P2). Havendo ao menos 1 mês com leitura em cada período, o indicador é calculado sobre a média dos meses disponíveis.

---

####  Análise de Sensibilidade do Requisito de Histórico:
A validação técnica demonstrou que a métrica se mantém **altamente consistente e robusta** independentemente da exigência do número mínimo de meses com leitura para identificar UCs com GD que registraram queda de consumo superior a 30%:

* **Critério de Mínimo 1 Mês (Implementado):** `3.120` UCs com GD e queda > 30%
* **Exigência de Mínimo 6 Meses:** `3.111` UCs com GD e queda > 30%
* **Exigência de Mínimo 9 Meses:** `3.078` UCs com GD e queda > 30%
* **Exigência de 12 Meses Integrais:** `725` UCs com GD e queda > 30%

> **Nota Metodológica:** A regra com requisito mínimo de **1 mês** preserva a cobertura volumétrica do modelo (capturando ~3.120 UCs elegíveis) sem distorcer o viés analítico do indicador.

---

####  Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `id_uc` | String | Identificador único da Unidade Consumidora |
| `municipio` | String | Município da UC |
| `bairro` | String | Bairro da UC |
| `possui_gd` | String | Indica se possui geração distribuída ("Sim" / "Não") |
| `consumo_medio_inicial` | Double | Consumo médio mensal em kWh no P1 (2022-07 a 2023-06) |
| `consumo_medio_final` | Double | Consumo médio mensal em kWh no P2 (2025-07 a 2026-06) |
| `variacao_kwh` | Double | Variação absoluta de consumo em kWh |
| `variacao_pct` | Double | Variação percentual de consumo (%) — `NULL` apenas sem leituras no período |

In [0]:
# Seção 5.2: tabela 2 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: variacao_consumo_uc ===")

# 1. Validação de formato de competência YYYY-MM
if (
    df_leituras_uc_mes
    .filter(
        F.col("mes_referencia").isNull()
        | ~F.col("mes_referencia").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$")
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("df_leituras_uc_mes contém competências nulas ou fora do formato YYYY-MM.")

# 2. Validações de integridade e granularidade da fonte de leituras
assert_no_nulls(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# 3. P1: julho/2022 a junho/2023 (Média + Contagem de meses com consumo VÁLIDO)
df_p1 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2022-07", "2023-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_inicial"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p1")
    )
)

# 4. P2: julho/2025 a junho/2026 (Média + Contagem de meses com consumo VÁLIDO)
df_p2 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2025-07", "2026-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_final"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p2")
    )
)

# 5. Base Cadastral de Referência
df_variacao_base = gold_visao_360.select("id_uc", "municipio", "bairro", "possui_gd")
assert_unique(df_variacao_base, ["id_uc"], "gold_visao_360 para variação")

df_variacao_unida = (
    df_variacao_base
    .join(df_p1, "id_uc", "left")
    .join(df_p2, "id_uc", "left")
    .withColumn("municipio", F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado")))
    .withColumn("bairro", F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado")))
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd").cast("string"), F.lit("Não")))
    .withColumn("meses_com_leitura_p1", F.coalesce(F.col("meses_com_leitura_p1"), F.lit(0)))
    .withColumn("meses_com_leitura_p2", F.coalesce(F.col("meses_com_leitura_p2"), F.lit(0)))
)

# Expressão para variação percentual bruta (não arredondada para máxima precisão da flag)
variacao_pct_bruta = (
    (F.col("consumo_medio_final") - F.col("consumo_medio_inicial")) 
    / F.col("consumo_medio_inicial")
) * 100

# 6. Construção das métricas, arredondamentos finais e flag booleana
gold_variacao_consumo_uc = (
    df_variacao_unida
    .withColumn(
        "variacao_kwh",
        F.when(
            F.col("consumo_medio_inicial").isNotNull() & F.col("consumo_medio_final").isNotNull(),
            F.round(F.col("consumo_medio_final") - F.col("consumo_medio_inicial"), 2)
        )
    )
    .withColumn(
        "variacao_pct_bruta",
        F.when(
            (F.col("consumo_medio_inicial") > 0) & F.col("consumo_medio_final").isNotNull(),
            variacao_pct_bruta
        )
    )
    .withColumn(
        "variacao_pct",
        F.round("variacao_pct_bruta", 2)
    )
    .withColumn(
        "queda_consumo_superior_30_pct",
        F.coalesce(F.col("variacao_pct_bruta") < -30, F.lit(False))
    )
    .drop("variacao_pct_bruta")
    .withColumn("consumo_medio_inicial", F.round("consumo_medio_inicial", 2))
    .withColumn("consumo_medio_final", F.round("consumo_medio_final", 2))
)

# 7. Asserções finais de saída
assert_no_nulls(gold_variacao_consumo_uc, ["id_uc", "municipio", "bairro", "possui_gd"], "variacao_consumo_uc")
assert_unique(gold_variacao_consumo_uc, ["id_uc"], "variacao_consumo_uc")

# 8. Persistência no catálogo Delta Lake
(
    gold_variacao_consumo_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
)

print("Tabela variacao_consumo_uc gravada com sucesso no catálogo Gold.")

# 9. Exibe amostra da tabela publicada lida diretamente do catálogo Delta
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
         .orderBy("id_uc")
         .limit(10)
)

####  Registros de Auditoria e Casos de Uso (GenAI) — Tabela 2

 * **Cobertura Base:** A tabela abrange exatamente **45.000 Unidades Consumidoras**, cobrindo 100% da base cadastral da distribuidora.
 * **Períodos Auditados:**
   - **P1 (Média Inicial):** Período de 2022-07 a 2023-06 (12 meses).
   - **P2 (Média Final):** Período de 2025-07 a 2026-06 (12 meses).
 * **Evidência do Caso de Uso (Issue #40):**
   - Foram mapeadas **3.120 UCs** que possuem Geração Distribuída (`possui_gd = 'Sim'`) e apresentaram queda de consumo superior a **30%** (variação percentual bruta estritamente menor que `-30.0%`).
 * **Tratamento de Exceções:** As médias usam os meses disponíveis em cada janela. Sem uma média, `variacao_kwh` fica nula; `variacao_pct` também fica nula quando a média inicial não é positiva. Os cálculos usam médias brutas e são arredondados a duas casas na projeção.
 * **Limite da evidência:** os resultados numéricos acima são observações do lote relatado; esta revisão de texto não renova o aceite nem comprova integração com backend ou IA.

### Tabela 3: Consumo e Receita Mensal por Unidade Consumidora (`gold_consumo_mensal_uc`)

**Objetivo:**  
Armazenar a série histórica temporal granular de consumo (`kWh`) e receita (`R$`) por Unidade Consumidora (UC) e competência (`mês/ano`). Esta tabela serve como camada intermediária para alimentar a **Visão 360°** e permitir que agentes de GenAI respondam a consultas pontuais de histórico de faturamento e consumo mensal por UC sem necessidade de reprocessar faturas em tempo de execução.

**Especificações da Tabela:**
* **Grão:** Unidade Consumidora (`id_uc`) × Competência (`competencia` / `YYYY-MM`).
* **Chave Primária:** (`id_uc`, `competencia`).
* **Regras de Negócio e Enriquecimento:**
  * **Junção de Fontes:** `INNER JOIN` entre faturas e leituras pré-agregadas por UC/mês, garantindo 100% de alinhamento com a `gold_diagnostico_receita`.
  * **Arredondamento:** Valores monetários (`receita_reais`) e volumétricos (`consumo_kwh`) padronizados em duas casas decimais.
  * **Persistência:** Tabela gravada no catálogo Delta Lake (`saveAsTable`) para rápida indexação e consulta SQL.

In [0]:
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA 3: consumo_mensal_uc ===")

# 1. Validações de integridade nas fontes de entrada
assert_no_nulls(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")
assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")

assert_no_nulls(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")
assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")

# 2. Junção das leituras e faturas por UC e Competência
# Inner Join mantém o alinhamento perfeito de faturamento com o diagnóstico financeiro
gold_consumo_mensal_uc = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "inner")
    .select(
        F.col("id_uc"),
        F.col("mes_referencia").alias("competencia"),
        F.round(F.col("consumo_uc_mes"), 2).alias("consumo_kwh"),
        F.round(F.col("receita_uc_mes"), 2).alias("receita_reais")
    )
)

# 3. Asserções de integridade
assert_no_nulls(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")
assert_unique(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")

# 4. Gravação no Catálogo Delta
tabela_t3_target = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
(
    gold_consumo_mensal_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_t3_target)
)

print(f" Tabela 3 'consumo_mensal_uc' gravada com sucesso em {tabela_t3_target}!")

# 5. Exibe amostra da tabela gravada
display(
    spark.table(tabela_t3_target)
         .orderBy("id_uc", "competencia")
         .limit(10)
)

# NOVAS TABELAS 
### apagarei essa celula depois

## 7. Camada Gold — Tabela: `faturas_uc`

### 1. Descrição e Objetivo
A tabela **`faturas_uc`** consolida o histórico individual de faturas associadas a cada Unidade Consumidora (UC)[cite: 8]. O objetivo principal é fornecer os detalhes financeiros e operacionais para a jornada **WEB-03 – Visão 360 da UC** sem depender da existência de leituras no mesmo mês[cite: 8].

---

### 2. Regras de Negócio e Qualidade
* **Grão Físico:** Uma linha por fatura (`id_fatura`)[cite: 8].
* **Tratamento de Datas Sentinelas e Futuras:** Datas de emissão, vencimento ou pagamento marcadas na Silver como sentinelas (`< 1950`) ou no futuro são convertidas para `NULL`[cite: 8, 9, 10].
* **Integridade Financeira:** A quitação e a presença de pagamento não são assumidas arbitrariamente; são validadas através das flags de quitação e inconsistência oriundas da Silver.
* **Preservação do Histórico:** Mantem todas as faturas do histórico da UC, independentemente do status (`PAGA`, `EM ABERTO`, etc.).

---

### 3. Especificação do Schema (`faturas_uc`)

| Coluna | Tipo | Origem Silver | Descrição / Regra |
| :--- | :--- | :--- | :--- |
| `id_fatura` | `STRING` | `id_fatura` | **Chave Primária**. Identificador único da fatura[cite: 8, 9]. |
| `id_uc` | `STRING` | `id_uc` | **Chave Estrangeira**. Vínculo com a Unidade Consumidora[cite: 8, 9]. |
| `mes_referencia` | `STRING` | `mes_referencia` | Competência do faturamento (formato `YYYY-MM`)[cite: 8, 9]. |
| `data_emissao` | `DATE` | `data_emissao` | Data de emissão (convertida para `NULL` se for sentinela ou futura)[cite: 8, 9, 10]. |
| `data_vencimento` | `DATE` | `data_vencimento` | Data de vencimento (convertida para `NULL` se for sentinela ou futura)[cite: 8, 9, 10]. |
| `data_pagamento` | `DATE` | `data_pagamento` | Data de quitação (convertida para `NULL` se for sentinela, futura ou data inválida)[cite: 8, 9, 10]. |
| `valor_energia` | `DECIMAL(20,2)` | `valor_energia` | Valor relativo ao consumo de energia elétrica[cite: 9]. |
| `tributos` | `DECIMAL(20,2)` | `tributos` | Valor cobrado referente a impostos e tributos[cite: 9]. |
| `valor_total` | `DECIMAL(20,2)` | `valor_total` | Valor total consolidado da fatura[cite: 9]. |
| `bandeira_tarifaria` | `STRING` | `bandeira_tarifaria` | Bandeira tarifária cobrada na competência[cite: 9]. |
| `status` | `STRING` | `status` | Situação declarada da fatura (`PAGA`, `EM ABERTO`, etc.)[cite: 8, 9]. |
| `flag_paga` | `BOOLEAN` | `_paga` | Evidência conclusiva de quitação do débito[cite: 9]. |
| `flag_tem_pagamento` | `BOOLEAN` | `_tem_pagamento` | Sinaliza existência de registro financeiro de pagamento[cite: 10]. |
| `flag_valor_inconsistente` | `BOOLEAN` | `_valor_inconsistente` | Alerta de divergência nos cálculos financeiros da fatura[cite: 9]. |
| `flag_data_pagamento_inconclusiva` | `BOOLEAN` | `_data_pagamento_inconclusiva` | Indica incerteza ou incoerência na data do pagamento[cite: 10]. |
| `data_processamento_gold` | `TIMESTAMP` | *Gerado na Gold* | Carimbo de data/hora do processamento do lote na camada Gold. |

---



In [0]:
%sql
SELECT * FROM silver_publicada.faturas LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE EXTENDED silver_publicada.faturas;

In [0]:
from pyspark.sql import functions as F

# 1. Leitura da camada Silver publicada
df_faturas_silver = spark.table("silver_publicada.faturas")

# 2. Construção da projeção Gold com saneamento e tratamento de sentinelas/futuros
df_faturas_gold = df_faturas_silver.select(
    F.col("id_fatura").cast("string"),
    F.col("id_uc").cast("string"),
    F.col("mes_referencia").cast("string"),
    
    # Tratamento de datas: se for marcada como sentinela ou futura, converte para NULL
    F.when(
        F.col("_emissao_sentinela") | F.col("_emissao_futura"), None
    ).otherwise(F.col("data_emissao")).alias("data_emissao"),
    
    F.when(
        F.col("_vencimento_sentinela") | F.col("_vencimento_futura"), None
    ).otherwise(F.col("data_vencimento")).alias("data_vencimento"),
    
    F.when(
        F.col("_pagamento_sentinela") | F.col("_pagamento_futura") | F.col("_paga_sem_data_valida"), None
    ).otherwise(F.col("data_pagamento")).alias("data_pagamento"),
    
    # Valores financeiros
    F.col("valor_energia").cast("decimal(20,2)"),
    F.col("tributos").cast("decimal(20,2)"),
    F.col("valor_total").cast("decimal(20,2)"),
    F.col("bandeira_tarifaria").cast("string"),
    F.col("status").cast("string"),
    
    # Flags de qualidade essenciais para a interpretação de negócio
    F.col("_paga").alias("flag_paga"),
    F.col("_tem_pagamento").alias("flag_tem_pagamento"),
    F.col("_valor_inconsistente").alias("flag_valor_inconsistente"),
    F.col("_data_pagamento_inconclusiva").alias("flag_data_pagamento_inconclusiva"),
    
    # Metadados de linhagem Gold
    F.current_timestamp().alias("data_processamento_gold")
)

# 3. Salva ou regista a tabela/visão na camada Gold
df_faturas_gold.createOrReplaceTempView("faturas_uc")

# 4. Grava a tabela física no catálogo Gold (workspace.gold.faturas_uc)
(
    df_faturas_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
)

print(f" Tabela {CATALOG}.{SCHEMA_GOLD}.faturas_uc salva com sucesso no Delta Lake!")

In [0]:
display(spark.sql("SELECT * FROM faturas_uc LIMIT 10"))

###  Suíte de Testes e Validação de Qualidade — `faturas_uc`

#### 1. Objetivo dos Testes
Garantir que a transformação PySpark aplicada à camada Gold atende a **100% dos critérios de aceite e requisitos de negócio** para a projeção `faturas_uc`, validando volumetria, integridade de chaves, saneamento de datas e consistência de atributos obrigatórios.

---

#### 2. Escopo das Validações

| ID | Teste / Validação | Critério de Aceite | Objetivo de Negócio |
| :---: | :--- | :--- | :--- |
| **01** | **Reconciliação Volumétrica** | `Volume Silver == Volume Gold` | Garantir que nenhuma fatura foi omitida ou duplicada indevidamente durante o processo de carga. |
| **02** | **Unicidade de Chave Primária** | `Chaves Duplicadas == 0` | Assegurar a integridade do grão físico de **uma linha por `id_fatura`**. |
| **03** | **Saneamento de Datas** | `Datas Incorretas == 0` | Confirmar que datas sentinelas (`< 1950`) ou futuras foram convertidas para `NULL` sem afetar registros válidos. |
| **04** | **Integridade de Atributos Obrigatórios** | `Registros com Nulos Proibidos == 0` | Garantir a ausência de nulos em identificadores chave (`id_fatura`, `id_uc`), competência e valores financeiros. |

---



In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Execução das verificações de qualidade e reconciliação
count_silver = spark.table("silver_publicada.faturas").count()
count_gold = spark.table("faturas_uc").count()

duplicados = spark.sql("""
    SELECT id_fatura FROM faturas_uc GROUP BY id_fatura HAVING COUNT(*) > 1
""").count()

datas_invalidas = spark.sql("""
    SELECT id_fatura FROM faturas_uc
    WHERE (data_emissao < '1950-01-01' OR data_emissao > CURRENT_DATE())
       OR (data_vencimento < '1950-01-01' OR data_vencimento > CURRENT_DATE())
       OR (data_pagamento < '1950-01-01' OR data_pagamento > CURRENT_DATE())
""").count()

nulos = spark.sql("""
    SELECT id_fatura FROM faturas_uc
    WHERE id_fatura IS NULL OR id_uc IS NULL OR mes_referencia IS NULL OR valor_total IS NULL
""").count()

# 2. Consolidação dos resultados em um relatório tabular
resultados = [
    ("1. Reconciliação Volumétrica (Silver vs Gold)", 
     "PASS" if count_silver == count_gold else "FAIL", 
     abs(count_silver - count_gold), 
     f"Silver: {count_silver} | Gold: {count_gold}"),
    
    ("2. Unicidade da Chave Primária (id_fatura)", 
     "PASS" if duplicados == 0 else "FAIL", 
     duplicados, 
     "Nenhum id_fatura duplicado encontrado" if duplicados == 0 else f"{duplicados} chaves duplicadas"),
    
    ("3. Saneamento de Datas (< 1950 e Futuras)", 
     "PASS" if datas_invalidas == 0 else "FAIL", 
     datas_invalidas, 
     "Todas as datas foram saneadas para NULL" if datas_invalidas == 0 else f"{datas_invalidas} datas inválidas não tratadas"),
    
    ("4. Integridade de Campos Obrigatórios", 
     "PASS" if nulos == 0 else "FAIL", 
     nulos, 
     "Nenhum atributo essencial nulo" if nulos == 0 else f"{nulos} registros com nulos proibidos")
]

schema_testes = StructType([
    StructField("Teste / Validação", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Inconformidades", IntegerType(), True),
    StructField("Detalhe", StringType(), True)
])

# 3. Exibição da tabela visual nativa no Databricks
df_relatorio_testes = spark.createDataFrame(resultados, schema=schema_testes)
display(df_relatorio_testes)

FIM DE NOVAS TABELAS 

## 7. Testes de Qualidade Automatizados e Validação do Pipeline
### Objetivo:
Garantir que os dados gerados na Gold atendem às restrições de integridade, contagem e consistência lógica antes de serem exportados para o banco de dados local da aplicação.

### 7.1 Suíte de Testes Automatizados de Qualidade e Integridade da Camada Gold

**Objetivo:** Executar a validação sistêmica de qualidade, unicidade de chaves primárias (PKs), não-nulidade e consistência de domínios em todas as **8 tabelas** consolidadas na camada Gold, atuando como a primeira trava do *Quality Gate*.

---

#### Escopo de Validação e Asserções Aplicadas:

1. **Mapeamento de Chaves Primárias (PKs Simples e Compostas):**
   * `diagnostico_receita`: `['competencia', 'municipio', 'classe_consumo', 'subgrupo_tarifario', 'bandeira_tarifaria', 'possui_gd']`
   * `indicadores_dec_fec`: `['id_alimentador', 'competencia']`
   * `fila_inspecao`: `['id_uc']`
   * `visao_360_uc`: `['id_uc']`
   * `faturas_uc`: `['id_fatura']`
   * `consumo_mensal_uc`: `['id_uc', 'competencia']`
   * `consumo_receita_bairro`: `['competencia', 'municipio', 'bairro', 'classe_consumo']`
   * `variacao_consumo_uc`: `['id_uc']`

2. **Verificações Estruturais e Integridade de Registros:**
   * **Presença de Dados:** Garantia de que nenhuma tabela da camada Gold está vazia (`count > 0`).
   * **Validação de Nulos em Chaves (`assert_no_nulls`):** Asserção estrita de que nenhum atributo formador de PK contenha valores `NULL`.
   * **Unicidade de Chaves Primárias (`assert_unique`):** Garantia de inexistência de duplicatas no grão definido de cada tabela.

3. **Regras de Negócio e Consistência de Domínio:**
   * **Integridade das Faturas (`faturas_uc`):** Obrigatoriedade de valores válidos nos campos vitais de negócio (`id_uc`, `mes_referencia`, `valor_total`) e validação estrita do formato de competência (`YYYY-MM`).
   * **Indicadores Operacionais (`indicadores_dec_fec`):** Bloqueio de valores negativos em DEC, FEC, durações e frequências de interrupção, além de garantia de denominadores válidos (`total_ucs_alimentador > 0`).
   * **Métricas Financeiras (`diagnostico_receita`):** Impedimento de receita ou consumo negativos e validação da tarifa média efetiva para consumos positivos.
   * **Conformidade Temporal:** Rigor no padrão regex para competências/meses de referência no formato `YYYY-MM` (`^[0-9]{4}-(0[1-9]|1[0-2])$`).
   * **Grão Único por UC:** Confirmação de que as visões por cliente (`fila_inspecao`, `visao_360_uc` e `variacao_consumo_uc`) mantêm rigorosamente a relação de 1 linha por `id_uc`.

In [0]:
# SEÇÃO 7.1 : Suíte de Testes Automatizados de Qualidade

from pyspark.sql import functions as F

# Dicionário atualizado cobrindo TODAS as tabelas da camada Gold (incluindo faturas_uc)
tables_gold_pk = {
    "diagnostico_receita": [
        "competencia",
        "municipio",
        "classe_consumo",
        "subgrupo_tarifario",
        "bandeira_tarifaria",
        "possui_gd"
    ],
    "indicadores_dec_fec": [
        "id_alimentador",
        "competencia"
    ],
    "fila_inspecao": ["id_uc"],
    "visao_360_uc": ["id_uc"],
    "faturas_uc": ["id_fatura"],  # <--- Nova tabela adicionada com sua PK
    "consumo_mensal_uc": ["id_uc", "competencia"],
    "consumo_receita_bairro": ["competencia", "municipio", "bairro", "classe_consumo"],
    "variacao_consumo_uc": ["id_uc"]
}

# Confere que todas as tabelas existem e têm registros.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_columns_exist(df, pk_cols, nome_completo)

    if df.limit(1).count() == 0:
        raise ValueError(f"{nome_completo} está vazia.")

    print(f"[PASS] {nome_completo}: possui registros.")

# Chaves compostas e primárias não podem conter nulos.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_no_nulls(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave sem nulos.")

# A chave declarada deve ser única.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_unique(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave única.")

# --- VALIDAÇÕES ESPECÍFICAS DA NOVA TABELA: faturas_uc ---
df_faturas = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")

# Atributos vitais não podem conter nulos
assert_no_nulls(df_faturas, ["id_uc", "mes_referencia", "valor_total"], f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
print("[PASS] faturas_uc: id_uc, mes_referencia e valor_total sem nulos.")

# Mês de referência no formato estrito YYYY-MM
mes_ref_invalido = (
    df_faturas
    .filter(
        F.col("mes_referencia").isNull()
        | ~F.col("mes_referencia").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$")
    )
    .limit(1)
    .count()
)

if mes_ref_invalido > 0:
    raise ValueError("faturas_uc contém mes_referencia fora do formato YYYY-MM.")

print("[PASS] faturas_uc: mes_referencia no formato YYYY-MM.")

# --- DEMAIS TESTES DE MÉTRICAS DAS OUTRAS TABELAS ---

# Verificar métricas negativas em DEC/FEC.
df_dec = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
)

if (
    df_dec
    .filter(
        (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
        | (F.col("frequencia_total_interrupcoes") < 0)
        | (F.col("total_ucs_alimentador") <= 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec contém valores negativos "
        "ou denominadores inválidos."
    )

# DEC e FEC não devem ser nulos nos grupos gerados.
if (
    df_dec
    .filter(F.col("DEC").isNull() | F.col("FEC").isNull())
    .limit(1)
    .count() > 0
):
    raise ValueError("indicadores_dec_fec contém DEC ou FEC nulos.")

print("[PASS] DEC/FEC sem valores negativos, nulos ou denominadores inválidos.")

# Verificar métricas de receita e consumo.
df_receita = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
)

if (
    df_receita
    .filter(
        (F.col("receita_total_reais") < 0)
        | (F.col("consumo_total_kwh") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita contém receita ou consumo negativos. "
        "Confirme se estornos e ajustes devem ser permitidos."
    )

# A tarifa efetiva pode ser nula quando consumo total é zero ou nulo.
if (
    df_receita
    .filter(
        (F.col("consumo_total_kwh") > 0)
        & F.col("tarifa_media_efetiva").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita tem tarifa efetiva nula "
        "apesar de consumo positivo."
    )

print("[PASS] Receita e consumo dentro dos limites definidos.")

# Formato estrito YYYY-MM para diagnostico_receita.
competencia_invalida = (
    df_receita
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_invalida > 0:
    raise ValueError(
        "diagnostico_receita contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de receita no formato YYYY-MM.")

# Checagem equivalente para DEC/FEC.
competencia_dec_invalida = (
    df_dec
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_dec_invalida > 0:
    raise ValueError(
        "indicadores_dec_fec contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de DEC/FEC no formato YYYY-MM.")

# Garantir que nenhuma UC foi multiplicada nas tabelas de visão/fila/variação.
for tabela in ["fila_inspecao", "visao_360_uc", "variacao_consumo_uc"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

    assert_unique(df, ["id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

print("[PASS] Uma linha por UC na fila, visão 360 e variação de consumo.")
print("Todos os testes de qualidade da Gold foram concluídos.")

### 7.2 Testes de Qualidade e Integridade Cruzada 

**Objetivo:** Validar a consistência física, financeira, cadastral e o saneamento temporal entre as tabelas analíticas complementares (`consumo_receita_bairro`, `variacao_consumo_uc`, `consumo_mensal_uc`, `faturas_uc`) e a tabela mestra de receitas (`diagnostico_receita`).

---

#### Escopo de Testes e Validações Automatizadas:

1. **Validação Estrutural e Integridade de Schemas:**
   * Confirmar a presença de todas as colunas obrigatórias nas tabelas analíticas e na projeção granular `faturas_uc` (`id_fatura`, `id_uc`, `mes_referencia`, `data_emissao`, `data_vencimento`, `data_pagamento`, `valor_total`, `status`, `flag_paga`).
   * Assegurar a não-nulidade e a unicidade das chaves primárias e dimensões de agrupamento (`bairro`, `id_uc`, `competencia`, `id_fatura`).

2. **Auditoria de Volumetria Cadastral e Saneamento Temporal:**
   * **Volumetria de UCs:** Garantir que a contagem distinta de Unidades Consumidoras na tabela de variação temporal (`variacao_consumo_uc`) seja 100% equivalente ao total de UCs operacionais retidas na `visao_360_uc` (`EXPECTED_UC_COUNT`).
   * **Saneamento de Datas em `faturas_uc`:** Asserção estrita para verificar que zero datas sentinelas (`< 1950-01-01`) ou futuras persistiram na camada Gold, confirmando a conversão para `NULL` conforme regramento de qualidade.

3. **Conciliação Físico-Financeira Cruzada (Tolerância: R$ 0,01 e 0,01 kWh):**
   * **Tabela Territorial (`consumo_receita_bairro`) vs. Diagnóstico:** A soma total da receita e do consumo por bairro deve bater exatamente com os totais consolidados no diagnóstico de receita.
   * **Tabela Mensal (`consumo_mensal_uc`) vs. Diagnóstico:** A consolidação do consumo e da receita individualizada por UC em nível mensal deve resultar no mesmo valor global do diagnóstico.

4. **Validação de Regra de Negócio Dinâmica (Geração Distribuída / Solar):**
   * Teste de asserção para comprovar a identificação das UCs equipadas com micro/minigeração distribuída (`possui_gd = 'Sim'`) que registraram redução expressiva de consumo (`queda_consumo_superior_30_pct = True`).

In [0]:
# SEÇÃO 7.2 : Testes de Qualidade e Integridade Cruzada

from pyspark.sql import functions as F

print("=== TESTES DE QUALIDADE E INTEGRIDADE CRUZADA ===")

# Nome das tabelas gravadas no catálogo Delta
TABELA_BAIRRO = f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro"
TABELA_VARIACAO = f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc"
TABELA_CONSUMO_MENSAL = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
TABELA_DIAGNOSTICO = f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
TABELA_VISAO_360 = f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc"
TABELA_FATURAS = f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc"

# Tolerâncias financeiras e operacionais
TOLERANCIA_REAIS = 0.01
TOLERANCIA_KWH = 0.01

# Carregar tabelas do catálogo Delta
df_bairro_test = spark.table(TABELA_BAIRRO)
df_variacao_test = spark.table(TABELA_VARIACAO)
df_mensal_test = spark.table(TABELA_CONSUMO_MENSAL)
df_diagnostico_test = spark.table(TABELA_DIAGNOSTICO)
df_v360_test = spark.table(TABELA_VISAO_360)
df_faturas_test = spark.table(TABELA_FATURAS)

# 1. Volumetria cadastral esperada via contagem distinta na Visão 360
EXPECTED_UC_COUNT = df_v360_test.select("id_uc").distinct().count()

# 2. Validação de Schemas
cols_esperadas_bairro = ["competencia", "municipio", "bairro", "classe_consumo", "consumo_total_kwh", "receita_total_reais", "qtd_ucs_distintas_faturadas_no_mes"]
cols_esperadas_variacao = ["id_uc", "municipio", "bairro", "possui_gd", "consumo_medio_inicial", "consumo_medio_final", "meses_com_leitura_p1", "meses_com_leitura_p2", "variacao_kwh", "variacao_pct", "queda_consumo_superior_30_pct"]
cols_esperadas_mensal = ["id_uc", "competencia", "consumo_kwh", "receita_reais"]
cols_esperadas_faturas = ["id_fatura", "id_uc", "mes_referencia", "data_emissao", "data_vencimento", "data_pagamento", "valor_total", "status", "flag_paga"]

assert_columns_exist(df_bairro_test, cols_esperadas_bairro, TABELA_BAIRRO)
assert_columns_exist(df_variacao_test, cols_esperadas_variacao, TABELA_VARIACAO)
assert_columns_exist(df_mensal_test, cols_esperadas_mensal, TABELA_CONSUMO_MENSAL)
assert_columns_exist(df_faturas_test, cols_esperadas_faturas, TABELA_FATURAS)
print("[PASS] Schemas das tabelas analíticas e da faturas_uc conferem.")

# 3. Nulos e Unicidade
chave_bairro = ["competencia", "municipio", "bairro", "classe_consumo"]
chave_mensal = ["id_uc", "competencia"]

assert_no_nulls(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_no_nulls(df_variacao_test, ["id_uc", "municipio", "bairro", "possui_gd"], TABELA_VARIACAO)
assert_no_nulls(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)
assert_no_nulls(df_faturas_test, ["id_fatura", "id_uc", "mes_referencia", "valor_total"], TABELA_FATURAS)

assert_unique(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_unique(df_variacao_test, ["id_uc"], TABELA_VARIACAO)
assert_unique(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)
assert_unique(df_faturas_test, ["id_fatura"], TABELA_FATURAS)
print("[PASS] Chaves e dimensões obrigatórias validadas sem nulos e únicas em todas as tabelas.")

# 4. Volumetria Cadastral em Variacao UC
total_ucs_variacao = df_variacao_test.select("id_uc").distinct().count()
assert total_ucs_variacao == EXPECTED_UC_COUNT, f"Volumetria divergente: esperado {EXPECTED_UC_COUNT:,}, encontrado {total_ucs_variacao:,}."
print(f"[PASS] Volumetria cadastral cravada em {total_ucs_variacao:,} UCs distintas.")

# 5. Validação de Saneamento de Datas em faturas_uc (< 1950 e Futuras)
datas_invalidas_faturas = (
    df_faturas_test
    .filter(
        (F.col("data_emissao") < "1950-01-01") | (F.col("data_emissao") > F.current_date())
        | (F.col("data_vencimento") < "1950-01-01") | (F.col("data_vencimento") > F.current_date())
        | (F.col("data_pagamento") < "1950-01-01") | (F.col("data_pagamento") > F.current_date())
    )
    .count()
)
assert datas_invalidas_faturas == 0, f"Encontradas {datas_invalidas_faturas} datas sentinelas/futuras não tratadas em faturas_uc!"
print("[PASS] Saneamento de datas em faturas_uc verificado: zero datas sentinelas ou futuras.")

# 6. Conciliação Físico-Financeira
totais_bairro = df_bairro_test.agg(
    F.sum("receita_total_reais").alias("receita_bairro"),
    F.sum("consumo_total_kwh").alias("consumo_bairro")
)

totais_mensal = df_mensal_test.agg(
    F.sum("receita_reais").alias("receita_mensal"),
    F.sum("consumo_kwh").alias("consumo_mensal")
)

totais_diagnostico = df_diagnostico_test.agg(
    F.sum("receita_total_reais").alias("receita_diagnostico"),
    F.sum("consumo_total_kwh").alias("consumo_diagnostico")
)

conciliacao_geral = (
    totais_bairro
    .crossJoin(totais_mensal)
    .crossJoin(totais_diagnostico)
    .withColumn("diff_receita_bairro", F.round(F.col("receita_bairro") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_bairro", F.round(F.col("consumo_bairro") - F.col("consumo_diagnostico"), 4))
    .withColumn("diff_receita_mensal", F.round(F.col("receita_mensal") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_mensal", F.round(F.col("consumo_mensal") - F.col("consumo_diagnostico"), 4))
)

display(conciliacao_geral)

diff_rec_bairro = conciliacao_geral.select("diff_receita_bairro").first()[0]
diff_kwh_bairro = conciliacao_geral.select("diff_consumo_bairro").first()[0]
diff_rec_mensal = conciliacao_geral.select("diff_receita_mensal").first()[0]
diff_kwh_mensal = conciliacao_geral.select("diff_consumo_mensal").first()[0]

assert abs(diff_rec_bairro) <= TOLERANCIA_REAIS, f"Falha na receita da Tabela Territorial: diferença de R$ {diff_rec_bairro}"
assert abs(diff_kwh_bairro) <= TOLERANCIA_KWH, f"Falha no consumo da Tabela Territorial: diferença de {diff_kwh_bairro} kWh"
assert abs(diff_rec_mensal) <= TOLERANCIA_REAIS, f"Falha na receita da Tabela Mensal: diferença de R$ {diff_rec_mensal}"
assert abs(diff_kwh_mensal) <= TOLERANCIA_KWH, f"Falha no consumo da Tabela Mensal: diferença de {diff_kwh_mensal} kWh"

print("[PASS] Conciliação Físico-Financeira ok para Tabelas Territorial e Mensal (diferenças zeradas em R$ e kWh).")

# 7. Validação Dinâmica de Regras de Negócio
queda_solar = (
    df_variacao_test
    .filter(
        (F.col("possui_gd") == "Sim")
        & (F.col("queda_consumo_superior_30_pct") == True)
    )
    .count()
)

print(f"UCs com GD e queda >30%: {queda_solar:,}")
assert queda_solar > 0, "ERRO: Nenhuma UC atendeu ao critério de GD com queda >30%!"

print(f"[PASS] Regra de Negócio validada dinamicamente: {queda_solar:,} UCs com Solar (GD) e queda >30% identificadas.")
print("\nTodos os testes de qualidade e integridade cruzada foram concluídos com 100% de sucesso!")

In [0]:
# DECOMPOSIÇÃO DE FATURAS DA SILVER 

from pyspark.sql import functions as F

faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
ucs = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")

# Normalização de colunas
col_comp_fat = "mes_referencia" if "mes_referencia" in faturas.columns else "competencia"
col_comp_lei = "mes_referencia" if "mes_referencia" in leituras.columns else "competencia"
col_val_fat = "valor_total" if "valor_total" in faturas.columns else "valor_total_reais"

# 1. Chaves de existência de Leituras
chaves_leituras = (
    leituras
    .filter(F.col("id_uc").isNotNull() & F.col(col_comp_lei).isNotNull())
    .select(
        F.col("id_uc"),
        F.col(col_comp_lei).alias("comp_leitura")
    )
    .dropDuplicates(["id_uc", "comp_leitura"])
)

# 2. Chaves de existência de Cadastro de UCs
chaves_ucs = (
    ucs
    .filter(F.col("id_uc").isNotNull())
    .select("id_uc")
    .dropDuplicates(["id_uc"])
)

# 3. Classificação Mutuamente Exclusiva das Faturas
faturas_classificadas = (
    faturas
    .filter(F.col("id_uc").isNotNull() & F.col(col_comp_fat).isNotNull())
    .join(
        chaves_leituras,
        (faturas["id_uc"] == chaves_leituras["id_uc"]) & 
        (F.col(col_comp_fat) == F.col("comp_leitura")),
        "left"
    )
    .withColumn("tem_leitura", F.col("comp_leitura").isNotNull())
    .join(chaves_ucs.withColumn("tem_cadastro", F.lit(True)), "id_uc", "left")
    .withColumn("tem_cadastro", F.coalesce(F.col("tem_cadastro"), F.lit(False)))
    .withColumn(
        "grupo",
        F.when(~F.col("tem_leitura"), "1. sem_leitura_correspondente")
         .when(~F.col("tem_cadastro"), "2. com_leitura_sem_cadastro")
         .otherwise("3. com_leitura_e_cadastro")
    )
)

print("--- DECOMPOSIÇÃO DE FATURAS DA SILVER ---")
display(
    faturas_classificadas
    .groupBy("grupo")
    .agg(
        F.count("*").alias("qtd_linhas_fatura"),
        F.sum(col_val_fat).alias("valor_total_reais")
    )
    .orderBy("grupo")
)

### 7.3 Reconciliação Financeira e Rastreabilidade do Descarte de Receita (Silver vs Gold)

**Motivação do Ajuste (Review de Engenharia):**  
Anteriormente, os testes de reconciliação limitavam-se ao `diagnostico_receita`, tabela que executa `INNER JOIN` entre faturas e leituras. Esse acoplamento descartava faturas emitidas sem medição pareada no mês, forçando uma correspondência parcial e ocultando a receita total retida na origem. Para sanar essa limitação e atender à visão 360 da Unidade Consumidora, foi introduzida a projeção **`faturas_uc`**, que preserva a totalidade das faturas independentemente da presença de leitura.

**Refatoração Aplicada na Suíte de Testes:**  
O teste de conciliação foi atualizado para executar duas asserções paralelas:
1. **Reconciliação Direta 1:1 (`faturas_uc`):** Validação estrita de que o valor bruto total e o volume de faturas da Silver foram 100% preservados na Gold sem qualquer perda centavo a centavo.
2. **Auditoria de Descarte Relacional (`diagnostico_receita`):** Comparação da receita bruta da Silver contra a receita integrada no Diagnóstico somada ao montante isolado de faturas sem leitura pareada.

---

#### Resultado da Auditoria Financeira e Reconciliação:

| Métrica / Projeção Gold | Volume de Faturas | Receita Retida (R$) | % Aproveitamento | Status da Reconciliação |
| :--- | :---: | :---: | :---: | :---: |
| **Base Bruta Silver (`faturas`)** | **2.091.968** | **R$ 647.055.875,02** | **100,00%** | Base de Origem |
| **Projeção Granular Gold (`faturas_uc`)** | **2.091.968** | **R$ 647.055.875,02** | **100,00%** | **PASS** (Divergência: R$ 0,00) |
| **Projeção Agregada Gold (`diagnostico_receita`)** | 1.928.920 | R$ 606.315.659,20 | 93,70% | Preservada via Inner Join |
| **Faturas Sem Leitura Correspondente** | 163.048 | R$ 40.740.215,82 | 6,30% | Auditada e Explicada |

---

#### Origem Técnica e Rastreabilidade da Divergência (Auditabilidade 100% Explicada)

A decomposição analítica comprova que a diferença entre a base bruta Silver e o `diagnostico_receita` decorre exclusivamente de regras de agrupamento de medição, sem atribuição a perdas operacionais não rastreadas:

1. **Preservação Integral em `faturas_uc`:**
   * **Volume:** 2.091.968 faturas | **Valor:** R$ 647.055.875,02 (100,00%)
   * **Causa:** A tabela `faturas_uc` foi modelada no grão individual de `id_fatura` sem dependência da tabela de leituras, garantindo a integridade histórica completa do faturamento.

2. **Faturas sem Leitura Correspondente (Descartadas no INNER JOIN do Diagnóstico):**
   * **Volume:** 163.048 faturas | **Valor:** R$ 40.740.215,82 (6,30%)
   * **Causa:** Como a construção do `diagnostico_receita` exige a relação `id_uc` + `mes_referencia` com a tabela de leituras, faturas emitidas sem registro de medição na mesma competência são desconsideradas dessa visão específica.

3. **Faturas sem Cadastro de UC (Órfãs):**
   * **Volume:** 0 faturas | **Valor:** R$ 0,00 (0,00%)
   * **Causa:** Inexistência de faturas atreladas a UCs fora da base cadastral oficial.

4. **Faturas Elegíveis Integradas no Diagnóstico:**
   * **Volume:** 1.928.920 registros | **Valor:** R$ 606.315.659,20 (93,70%)

---

#### Conclusão da Auditoria:
A reconciliação financeira comprova a integridade total da camada Gold: a projeção `faturas_uc` retém 100% da receita e do volume de faturas da Silver (diferença zerada em R$ 0,00), enquanto a variação do `diagnostico_receita` é 100% explicada e reconciliada pelas faturas sem leitura pareada: 

**Receita Silver = Diagnóstico Gold + Faturas Sem Leitura**

In [0]:
# 7.3 TESTE DE RECONCILIÇÃO FINANCEIRA E AUDITORIA DE DESCARTE (SILVER vs GOLD)

from pyspark.sql import functions as F

faturas_silver = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
gold_faturas_uc = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")  # <--- Tabela faturas_uc incluída

col_valor_silver = "valor_total_reais" if "valor_total_reais" in faturas_silver.columns else "valor_total"
col_val_fat = col_valor_silver

# 1. Total Bruto de Faturas na Silver (sem filtros)
silver_bruta_agg = faturas_silver.agg(
    F.sum(col_valor_silver).alias("receita_bruta_silver"),
    F.count("*").alias("qtd_faturas_silver"),
    F.countDistinct("id_uc").alias("total_ucs_cadastradas_silver")
)

# 2. Total na faturas_uc (Projeção Granular Gold)
gold_faturas_uc_agg = gold_faturas_uc.agg(
    F.sum("valor_total").alias("receita_faturas_uc"),
    F.count("*").alias("qtd_faturas_uc")
)

# 3. Total Processado e Média Mensal de UCs Faturadas no diagnostico_receita (Projeção Agregada Gold)
gold_receita_agg = gold_receita.agg(
    F.sum("receita_total_reais").alias("receita_gold_diagnostico"),
    F.round(F.avg("qtd_ucs_distintas_faturadas_no_mes"), 0).alias("media_mensal_ucs_gold")
)

# 4. Consolidação da Tabela de Reconciliação
reconciliacao_financeira = (
    silver_bruta_agg
    .crossJoin(gold_faturas_uc_agg)
    .crossJoin(gold_receita_agg)
    .select(
        F.col("receita_bruta_silver"),
        F.col("receita_faturas_uc"),
        F.col("receita_gold_diagnostico"),
        (F.col("receita_bruta_silver") - F.col("receita_gold_diagnostico")).alias("diferenca_descarte_reais"),
        F.round(((F.col("receita_gold_diagnostico") / F.col("receita_bruta_silver")) * 100), 2).alias("pct_aderencia_diagnostico"),
        F.col("qtd_faturas_silver"),
        F.col("qtd_faturas_uc"),
        F.col("total_ucs_cadastradas_silver"),
        F.col("media_mensal_ucs_gold")
    )
)

print("--- RESULTADO DA RECONCILIÇÃO FINANCEIRA E RASTREABILIDADE DE DESCARTE ---")
display(reconciliacao_financeira)

resumo = reconciliacao_financeira.collect()[0]
print(f"\n[AUDITORIA FINANCEIRA]")
print(f"• Receita Bruta Silver:                R$ {resumo['receita_bruta_silver']:,.2f}")
print(f"• Receita Granular Gold (faturas_uc):  R$ {resumo['receita_faturas_uc']:,.2f}")
print(f"• Receita Agregada Gold (diagnostico): R$ {resumo['receita_gold_diagnostico']:,.2f}")
print(f"• Receita Descartada no Diagnóstico:   R$ {resumo['diferenca_descarte_reais']:,.2f}")
print(f"• Taxa de Aproveitamento Diagnóstico:  {resumo['pct_aderencia_diagnostico']}%")

# --- ASSERÇÃO 1: RECONCILIÇÃO DIRETA 1:1 (faturas_silver vs faturas_uc) ---
diff_volumetrica_faturas = abs(resumo['qtd_faturas_silver'] - resumo['qtd_faturas_uc'])
diff_financeira_faturas = abs(resumo['receita_bruta_silver'] - resumo['receita_faturas_uc'])

assert diff_volumetrica_faturas == 0, (
    f" FALHA DE RECONCILIÇÃO VOLUMÉTRICA EM faturas_uc!\n"
    f"• Faturas Silver: {resumo['qtd_faturas_silver']:,} | faturas_uc: {resumo['qtd_faturas_uc']:,}"
)

assert diff_financeira_faturas <= 0.01, (
    f" FALHA DE RECONCILIÇÃO FINANCEIRA EM faturas_uc!\n"
    f"• Receita Silver: R$ {resumo['receita_bruta_silver']:,.2f} | faturas_uc: R$ {resumo['receita_faturas_uc']:,.2f}"
)

print(f"\n[PASS] Reconciliação 1:1 (Silver vs faturas_uc) cravada: {resumo['qtd_faturas_uc']:,} faturas e R$ {resumo['receita_faturas_uc']:,.2f} mantidos sem perdas.")

# --- ASSERÇÃO 2: RECONCILIÇÃO FÍSICO-FINANCEIRA DO DIAGNÓSTICO DE RECEITA ---
# Isola a soma do valor faturado das faturas classificadas como sem leitura
receita_sem_leitura = (
    faturas_classificadas
    .filter(F.col("grupo") == "1. sem_leitura_correspondente")
    .agg(F.sum(col_val_fat))
    .first()[0] or 0.0
)

# Reconciliação Independente: Faturas Silver = Diagnostico Gold + Faturas Sem Leitura
receita_total_explicada = resumo['receita_gold_diagnostico'] + receita_sem_leitura
diferenca_nao_explicada = abs(resumo['receita_bruta_silver'] - receita_total_explicada)

assert diferenca_nao_explicada <= 0.01, (
    f" FALHA CRÍTICA DE RECONCILIÇÃO FÍSICO-FINANCEIRA DO DIAGNÓSTICO!\n"
    f"• Receita Silver: R$ {resumo['receita_bruta_silver']:,.2f}\n"
    f"• Receita Explicada (Diagnóstico + Sem Leitura): R$ {receita_total_explicada:,.2f}\n"
    f"• Divergência Inexplicada: R$ {diferenca_nao_explicada:,.2f}"
)

print(f"[PASS] Reconciliação Físico-Financeira do Diagnóstico cravada! Divergência: R$ {diferenca_nao_explicada:.2f}")

###  7.4 Validação Dinâmica e Auditoria Rastreável do DEC/FEC (`indicadores_dec_fec`)

**Objetivo:** Garantir a exatidão matemática e a integridade relacional dos indicadores de continuidade e qualidade do fornecimento de energia (**DEC - Duração Equivalente de Interrupção por Unidade Consumidora** e **FEC - Frequência Equivalente**) na camada Gold.

---

####  Metodologia do Teste de Auditoria Operacional:
Para evitar asserções estáticas (hardcoded) e garantir que o pipeline seja auditável contra qualquer alteração na camada Silver, o teste reexecuta os cálculos de forma independente e dinâmica:

1. **Recálculo do Denominador Efetivo (Silver):** Quantifica o total exato de Unidades Consumidoras (`id_uc`) ativas no alimentador alvo (ex: `ALM-REC-013`), aplicando o mesmo filtro de exclusão de UCs sintéticas/teste (`id_uc NOT LIKE 'TESTE%'`).
2. **Apuração do Histórico de Interrupções (Silver):** Realiza o `INNER JOIN` entre o cadastro limpo de UCs do alimentador e a tabela `ocorrencias_rede` para a competência avaliada (ex: `2026-01`), somando a duração total das interrupções em horas.
3. **Cálculo da Métrica Esperada:** Apura o $\text{DEC Esperado} = \frac{\text{Duração Total de Interrupções (horas)}}{\text{Total de UCs no Alimentador}}$.
4. **Asserção Rígida (Gold vs Silver):** Confronta o registro gerado na tabela `indicadores_dec_fec` contra os valores calculados diretamente na Silver, validando a paridade de 100% no denominador (`total_ucs_alimentador`) e a exatidão do DEC até a 4ª casa decimal.

---

####  Resultado da Validação:
* **Alimentador Avaliado:** `ALM-REC-013`
* **Competência:** `2026-01`
* **Status da Auditoria:** **`[PASS]`** (Denominador e DEC 100% conciliados entre as camadas Silver e Gold, sem divergência relacional).


In [0]:

# 7.4 VALIDAÇÃO DINÂMICA DO DEC/FEC (ALM-REC-013)


from pyspark.sql import functions as F

alimentador_teste = "ALM-REC-013"
competencia_teste = "2026-01"

print(f" Validando dinamicamente DEC/FEC para {alimentador_teste} | Competência: {competencia_teste}")

# 1. Recalcula o denominador esperado diretamente da Silver (excluindo UCs 'TESTE')
ucs_silver_validas = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")
    .filter(F.col("id_uc").isNotNull() & ~F.col("id_uc").startswith("TESTE"))
    .filter(F.col("alimentador") == alimentador_teste)
    .select("id_uc")
    .distinct()
)

total_ucs_esperado = ucs_silver_validas.count()

# 2. Recalcula a duração esperada diretamente da Silver
ocorrencias_silver = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede")
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & (F.date_format(F.to_timestamp("data_abertura"), "yyyy-MM") == competencia_teste)
    )
    .join(ucs_silver_validas, "id_uc", "inner")
)

duracao_horas_esperada = (
    ocorrencias_silver.agg(
        F.sum(F.col("duracao_min").cast("double") / F.lit(60.0))
    ).collect()[0][0] or 0.0
)

# 3. Calcula o DEC esperado dinamicamente
dec_esperado = duracao_horas_esperada / total_ucs_esperado if total_ucs_esperado > 0 else 0.0

# 4. Lê a tabela Gold recém-gerada
df_dec_fec_gold = (
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec")
    .filter(
        (F.col("id_alimentador") == alimentador_teste)
        & (F.col("competencia") == competencia_teste)
    )
)

# 5. Assert de existência do registro na Gold
assert df_dec_fec_gold.count() == 1, (
    f"Erro: Registro para {alimentador_teste} na competência {competencia_teste} não foi encontrado na Gold!"
)

row = df_dec_fec_gold.first()

# 6. Assert dinâmico do Denominador (compara contra a Silver)
assert row["total_ucs_alimentador"] == total_ucs_esperado, (
    f"Divergência no denominador: esperado {total_ucs_esperado} (Silver), mas encontrado {row['total_ucs_alimentador']} (Gold)"
)

# 7. Assert dinâmico do DEC (compara contra a Silver)
dec_calculado = float(row["DEC"])
assert abs(dec_calculado - dec_esperado) < 0.0001, (
    f"Divergência no DEC do {alimentador_teste}: esperado ~{dec_esperado:.4f} (Silver), mas encontrado {dec_calculado:.4f} (Gold)"
)

print(f" [PASS] Validação dinâmica do {alimentador_teste} concluída com sucesso! (DEC: {dec_calculado:.4f} | Denominador: {row['total_ucs_alimentador']})")

# 8. Exibe os resultados validados na tela para conferência visual
display(df_dec_fec_gold)

### 7.5 Testes Complementares de Sanidade, Cobertura Temporal e Trava de Segurança (Fail-Fast)

**Objetivo:** Executar a checagem final de sanidade dos dados, saneamento de datas e integridade temporal entre as camadas Silver e Gold em todas as projeções (incluindo `faturas_uc`), atuando como o **mecanismo de interrupção imediata (Fail-Fast)** para impedir a persistência ou disponibilização de dados divergentes no catálogo.

---

#### Escopo de Testes e Validações de Sanidade:

1. **Asserção de Sanidade de Métricas (`diagnostico_receita` e `faturas_uc`):**
   * **Diagnóstico de Receita:** Verificação estrita contra valores nulos (`NULL`) ou numericamente inválidos (negativos) nas colunas de faturamento (`receita_total_reais` e `consumo_total_kwh`).
   * **Faturas da UC:** Garantia de preenchimento obrigatório dos campos vitais (`id_fatura`, `id_uc`, `mes_referencia`, `valor_total`) e bloqueio de montantes financeiros negativos.
   * **Saneamento de Datas:** Asserção de integridade em `faturas_uc` para confirmar a inexistência de datas sentinelas (`< 1950-01-01`) ou futuras não tratadas nos atributos de emissão, vencimento e pagamento.

2. **Auditoria de Cobertura Temporal (Silver vs Gold):**
   * **Preservação Temporal em `faturas_uc`:** Comparação do conjunto de meses de referência da Silver contra a Gold, garantindo 100% de cobertura temporal sem omissão de meses (`diff_meses_faturas_uc == set()`).
   * **Mapeamento de Competências no Diagnóstico:** Acompanhamento do comportamento temporal de `diagnostico_receita` em relação às competências com medição pareada.

3. **Volumetria Global da Visão 360 (`visao_360_uc`):**
   * Validação estrita no catálogo Delta para homologar a leitura e a contagem distinta de Unidades Consumidoras (`id_uc`) operacionais na tabela central do cliente.

4. **Mecanismo de Interrupção Rígida (Fail-Fast Pipeline Guard):**
   * Bloco condicional que consolida todas as divergências acumuladas e dispara uma exceção explícita (`raise ValueError`) ao identificar registros inválidos, datas corrompidas ou meses ausentes em `faturas_uc`.
   * **Garantia de Produção:** Cancelamento imediato do job de ETL para prevenir a exposição de dados inconsistentes às aplicações consumidoras e modelos analíticos.

In [0]:
# 7.5 TESTES COMPLEMENTARES: NULOS, TEMPORAL E VISÃO 360 (INCLUINDO faturas_uc)

from pyspark.sql import functions as F

print("--- RUNNING SANITY CHECKS ---")

schema_silver_ref = SCHEMA_SILVER_DADOS if 'SCHEMA_SILVER_DADOS' in locals() else SCHEMA_SILVER

# Carregamento das tabelas Gold e Silver para auditoria
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
gold_faturas = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.faturas_uc")
silver_faturas = spark.table(f"{CATALOG}.{schema_silver_ref}.faturas")

# 3.1 Teste de Nulos ou Métricas Inválidas (diagnostico_receita + faturas_uc)
invalidos_diagnostico = gold_receita.filter(
    F.col("receita_total_reais").isNull() |
    F.col("consumo_total_kwh").isNull() |
    (F.col("receita_total_reais") < 0) |
    (F.col("consumo_total_kwh") < 0)
).count()

invalidos_faturas_uc = gold_faturas.filter(
    F.col("id_fatura").isNull() |
    F.col("id_uc").isNull() |
    F.col("mes_referencia").isNull() |
    F.col("valor_total").isNull() |
    (F.col("valor_total") < 0)
).count()

# Teste de Saneamento de Datas em faturas_uc (< 1950 ou no futuro)
datas_invalidas_faturas = gold_faturas.filter(
    (F.col("data_emissao") < "1950-01-01") | (F.col("data_emissao") > F.current_date()) |
    (F.col("data_vencimento") < "1950-01-01") | (F.col("data_vencimento") > F.current_date()) |
    (F.col("data_pagamento") < "1950-01-01") | (F.col("data_pagamento") > F.current_date())
).count()

total_invalidos = invalidos_diagnostico + invalidos_faturas_uc + datas_invalidas_faturas

print(f"[TESTE 1.1] Métricas nulas ou inválidas no Diagnóstico: {invalidos_diagnostico} (Esperado: 0)")
print(f"[TESTE 1.2] Métricas nulas ou inválidas em faturas_uc: {invalidos_faturas_uc} (Esperado: 0)")
print(f"[TESTE 1.3] Datas sentinelas/futuras não tratadas em faturas_uc: {datas_invalidas_faturas} (Esperado: 0)")

# 3.2 Cobertura Temporal de Competências / Meses de Referência
col_silver = "competencia" if "competencia" in silver_faturas.columns else "mes_referencia"
col_gold_rec = "competencia" if "competencia" in gold_receita.columns else "mes_referencia"

comp_silver = set(r[col_silver] for r in silver_faturas.select(col_silver).distinct().collect() if r[col_silver])
comp_gold_rec = set(r[col_gold_rec] for r in gold_receita.select(col_gold_rec).distinct().collect() if r[col_gold_rec])
comp_gold_fat = set(r["mes_referencia"] for r in gold_faturas.select("mes_referencia").distinct().collect() if r["mes_referencia"])

diff_meses_diagnostico = comp_silver - comp_gold_rec
diff_meses_faturas_uc = comp_silver - comp_gold_fat

print(f"[TESTE 2.1] Competências ausentes no Diagnóstico (Esperado via Inner Join): {diff_meses_diagnostico}")
print(f"[TESTE 2.2] Meses de referência ausentes em faturas_uc: {diff_meses_faturas_uc} (Esperado: set())")

# 3.3 Cobertura de UCs na Visão 360 (Validação Estrita na Tabela Delta Persistida)
df_v360_prep = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc").select("id_uc").distinct()

try:
    total_v360 = df_v360_prep.count()
    print(f"[TESTE 3] Total de UCs únicas na Visão 360 (Tabela Persistida): {total_v360}")
except Exception as e:
    raise RuntimeError(
        f"FALHA CRÍTICA: Não foi possível ler a tabela 'visao_360_uc' diretamente do catálogo Gold. "
        f"Verifique se a persistência física ocorreu corretamente. Erro: {str(e)}"
    )

# INTERRUPÇÃO EXPLÍCITA EM CASO DE FALHA NO PIPELINE (FAIL-FAST)
total_divergencias = total_invalidos + len(diff_meses_faturas_uc)

if total_divergencias > 0:
    raise ValueError(
        f" FALHA NO PIPELINE DE DADOS: Foram encontradas {total_divergencias} divergências nos testes complementares "
        f"({total_invalidos} registros inválidos/nulos e {len(diff_meses_faturas_uc)} meses ausentes em faturas_uc). "
        f"O Job foi interrompido para evitar a publicação de dados inconsistentes na Camada Gold."
    )

print("\n TODOS OS TESTES PASSARAM COM SUCESSO! A CAMADA GOLD ESTÁ HOMOLOGADA COM 0 DIVERGÊNCIAS.")


## 8. Linhagem e Registro de Execução (`gold.controle_gold`)

**Objetivo:** Persistir os metadados de execução da camada Gold, assegurando a rastreabilidade completa e o vínculo estrito com o `run_id` e lote da camada Silver de origem.

---

### Diretrizes de Governança e Operação do Job:

1. **Rastreabilidade Histórica (`append`):**
   * A tabela `gold.controle_gold` acumula o histórico de execuções via `append`, registrando a chave `run_id_silver`, lote de origem, status de processamento, lista de tabelas geradas e o ambiente de execução (`prod` / `dev`).

2. **Diferenciação de Registros (Desenvolvimento vs. Produção):**
   * Execuções manuais do notebook em ambiente de desenvolvimento (`dev`) registram eventos locais para rastreio de testes.
   * **Registro Oficial de Auditoria:** Apenas as execuções orquestradas e automatizadas pelo **Job/Workflow oficial em ambiente de produção (`prod`)** são consideradas carimbos válidos de publicação e homologação dos dados para consumo.

3. **Status de Homologação:**
   * Grava o estado `"CONCLUIDO"` indicando o processamento e a aprovação bem-sucedida de todas as **8 tabelas Gold** na suíte de testes de qualidade.

4. **Tabelas Registradas no Lote (8 Tabelas):**
   * `diagnostico_receita`, `indicadores_dec_fec`, `fila_inspecao`, `visao_360_uc`, `faturas_uc`, `consumo_mensal_uc`, `consumo_receita_bairro` e `variacao_consumo_uc`.

In [0]:
import datetime
from uuid import uuid4
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

tb_controle_gold = f"{CATALOG}.{SCHEMA_GOLD}.controle_gold"

# Não usar fallback silencioso para ambiente ou metadados de linhagem.
if AMBIENTE not in {"dev", "test", "prod"}:
    raise RuntimeError(f"AMBIENTE não configurado corretamente: {AMBIENTE!r}")

if "run_id_silver" not in globals() or not run_id_silver:
    raise RuntimeError("run_id_silver ausente; registro Gold interrompido.")

if "lote_bronze_silver" not in globals() or not lote_bronze_silver:
    raise RuntimeError("lote_bronze_silver ausente; registro Gold interrompido.")

if "silver_lineage_status" not in globals() or not silver_lineage_status:
    raise RuntimeError(
        "silver_lineage_status ausente; registro Gold interrompido."
    )

# Em produção, só registrar conclusão se a Silver foi validada fisicamente.
if AMBIENTE == "prod":
    if silver_lineage_status != "CONTROLE_FISICO":
        raise RuntimeError(
            "Produção exige linhagem física da Silver. "
            f"Status recebido: {silver_lineage_status}"
        )

    if "status_exec" not in globals() or str(status_exec).strip().upper() != "CONCLUIDO":
        raise RuntimeError(
            "Produção exige status Silver CONCLUIDO."
        )

# Confere TODAS as 8 tabelas que esta execução declara ter gerado 
tabelas_geradas = [
    "diagnostico_receita",
    "indicadores_dec_fec",
    "fila_inspecao",
    "visao_360_uc",
    "faturas_uc",  #  Nova tabela adicionada 
    "consumo_mensal_uc",
    "consumo_receita_bairro",
    "variacao_consumo_uc"
]

tabelas_ausentes = [
    f"{CATALOG}.{SCHEMA_GOLD}.{nome}"
    for nome in tabelas_geradas
    if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_GOLD}.{nome}")
]

if tabelas_ausentes:
    raise RuntimeError(
        "Não foi possível registrar a Gold como concluída. "
        f"Tabelas ausentes: {tabelas_ausentes}"
    )

# Em ambiente de teste com linhagem sintética, o status Silver fica explícito.
status_silver_val = (
    str(status_exec).strip().upper()
    if "status_exec" in globals() and status_exec is not None
    else "NAO_VALIDADO"
)

run_id_gold = (
    f"GOLD_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

dados_controle = [(
    run_id_gold,
    str(run_id_silver),
    str(lote_bronze_silver),
    status_silver_val,
    str(silver_lineage_status),
    "CONCLUIDO",
    ", ".join(tabelas_geradas),
    AMBIENTE,
    datetime.datetime.now(datetime.timezone.utc)
)]

schema_controle = StructType([
    StructField("run_id_gold", StringType(), False),
    StructField("run_id_silver", StringType(), False),
    StructField("lote_bronze_silver", StringType(), False),
    StructField("status_silver", StringType(), False),
    StructField("silver_lineage_status", StringType(), False),
    StructField("status_gold", StringType(), False),
    StructField("tabelas_geradas", StringType(), False),
    StructField("ambiente", StringType(), False),
    StructField("processado_em", TimestampType(), False)
])

df_novo_registro = spark.createDataFrame(
    dados_controle,
    schema=schema_controle
)

if spark.catalog.tableExists(tb_controle_gold):
    (
        df_novo_registro.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(tb_controle_gold)
    )
else:
    (
        df_novo_registro.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tb_controle_gold)
    )

print(
    f"[PASS] Execução Gold registrada em {tb_controle_gold}: "
    f"run_id_gold={run_id_gold}, ambiente={AMBIENTE}, "
    f"status_silver={status_silver_val}, "
    f"linhagem={silver_lineage_status}."
)

## Revisão de semântica do dicionário — 28/10/2026

A média histórica usa leituras não nulas; variações são calculadas antes do arredondamento a duas casas e a flag usa variação bruta < −30%. DEC/FEC conta eventos por UC afetada e publica somente grupos com ocorrência. As categorias e a ausência de casos em ALTA - CONSUMO ZERO são observações de lote. Consulte docs/dicionario_gold.md e o registro de progresso do PRD. Esta revisão preserva todas as células de código e não reproduz a bateria externa de 122 verificações.